# **External Forecasting Signal Pipeline**

This workbook organises external data sources, turns the useful parts into monthly forecasting features, and stores searchable summaries so the team can ask what data exists and how it should be used.

## **Current Pipeline**

This is the short version to talk the team through: the notebook turns external sources into a monthly forecasting handoff table, and separately stores searchable context for LLM questions.



The forecasting models consume monthly_external_features; the vector database is for search and explanation.


## **Table of Contents**

1. **Libraries & Installs** - install and import the packages used by the pipeline.
2. **Helper Functions** - reusable file loaders and document builders.
3. **Source Registry** - the control table for every external source: what it is, how useful it is, and how it joins to forecasts.
4. **Usefulness Scoring Guide** - explains how the usefulness rating is decided.
5. **Load Registered Sources** - loads every active file source and keeps API candidates registered for discovery.
6. **Source Documents** - creates registry, summary and row-level documents for search/RAG.
7. **Source Preview** - quick human inspection of generated documents and tables.
8. **Feature Inventory** - audits source columns and maps raw column names to model-ready names.
9. **Forecasting Feature Bridge** - turns cleaned source tables into structured model features.
10. **Monthly Forecasting Bridge** - converts slow annual context into monthly, lagged, non-additive features.
11. **Monthly API Feature Sources** - builds monthly weather and calendar features from APIs.
12. **Monthly Feature Assembly** - combines external monthly features into one model handoff table.
13. **External Data EDA and Presentation Plots** - creates Plotly charts from the preprocessed external features and saves them as HTML artifacts.
14. **Quality Checks** - validates shape, keys, leakage assumptions and feature semantics.
15. **Export Model Handoff Files** - writes CSV/Parquet files the team can merge into forecasting models.
16. **Embed and Store** - embeds all source/search documents for retrieval.
17. **Embedding Sanity Checks** - confirms embeddings/storage count.
18. **Retrieval Smoke Tests** - checks that retrieval returns the expected source types.
19. **LLM Pipeline Q&A** - optional assistant for asking questions about sources, features, joins and limitations.


# **Libraries & Installs**

In [ ]:
# Installs
!pip install -qU \
    pandas \
    openpyxl \
    xlrd \
    pypdf \
    requests \
    plotly \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    langchain-huggingface \
    langchain-openai \
    sentence-transformers \
    unstructured \
    msoffcrypto-tool


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━

In [ ]:
# Libraries

from __future__ import annotations

from pathlib import Path
import time
import unicodedata
import requests
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


/tmp/ipykernel_7615/3317786045.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
#Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Helper Functions**

In [ ]:
# Convert each dataframe row into one LangChain Document

def dataframe_to_documents(
    dataframe,
    source_file,
    sheet_name,
    source_profile=None
):
    """
    Convert each dataframe row into one LangChain Document.
    """

    source_profile = source_profile or {}
    documents = []

    for row_number, (_, row) in enumerate(
        dataframe.iterrows(),
        start=2
    ):

        content_parts = []

        for column, value in row.items():

            if pd.notna(value):
                content_parts.append(
                    f"{column}: {value}"
                )

        page_content = "\n".join(content_parts)

        # Ignore completely empty rows
        if not page_content:
            continue

        metadata = {
            "source_file": Path(source_file).name,
            "file_type": Path(source_file).suffix.lower(),
            "source_type": "external_forecasting_signal",
            "record_type": "table_row",
            "sheet_name": str(sheet_name),
            "row_number": row_number
        }

        metadata.update(source_profile)

        if "Año" in row.index and pd.notna(row["Año"]):
            metadata["year"] = int(row["Año"])

        document = Document(
            page_content=page_content,
            metadata=metadata
        )

        documents.append(document)

    return documents

In [ ]:
#Formatting for excel docs
def flatten_column_names(columns):
    """
    Combine multi-row Excel headers into one clear column name.
    """

    clean_columns = []

    for column in columns:

        # Multi-row headers are returned as tuples
        if isinstance(column, tuple):
            column_parts = column
        else:
            column_parts = [column]

        clean_parts = []

        for part in column_parts:

            part = str(part).strip()

            # Ignore automatically generated Unnamed headings
            if not part.startswith("Unnamed"):
                clean_parts.append(part)

        clean_name = " - ".join(clean_parts)

        clean_columns.append(clean_name)

    return clean_columns

In [ ]:
# Function to load Excel, CSV or PDF files

def repair_mojibake_text(value):
    """
    Repair common UTF-8 text that has been decoded as Latin-1.
    Example: fixes filenames where accented characters were decoded incorrectly.
    """

    try:
        return str(value).encode("latin1").decode("utf-8")
    except UnicodeError:
        return str(value)


def normalise_match_text(value):
    """
    Normalise filenames so accented/non-accented variants can be compared.
    """

    value = unicodedata.normalize(
        "NFKD",
        str(value).lower()
    )

    value = value.encode(
        "ascii",
        "ignore"
    ).decode("ascii")

    return "".join(
        character
        for character in value
        if character.isalnum()
    )


def resolve_file_path(file_path):
    """
    Resolve a source path robustly in Colab/Drive.

    Tries:
    1. the path exactly as registered,
    2. a repaired accented version,
    3. a fuzzy search in the same folder.
    """

    original_path = Path(file_path)

    if original_path.exists():
        return original_path

    repaired_path = Path(
        repair_mojibake_text(file_path)
    )

    if repaired_path.exists():
        return repaired_path

    search_directories = []

    for candidate_path in [original_path, repaired_path]:
        if candidate_path.parent.exists():
            search_directories.append(
                candidate_path.parent
            )

    search_directories = list(dict.fromkeys(search_directories))

    suffixes = {
        original_path.suffix.lower(),
        repaired_path.suffix.lower()
    }

    desired_text = normalise_match_text(
        repaired_path.stem
    )

    key_tokens = [
        token
        for token in [
            "siniestro",
            "transito",
            "chile",
            "1972",
            "2024"
        ]
        if token in desired_text
    ]

    for search_directory in search_directories:

        candidate_files = [
            candidate
            for candidate in search_directory.iterdir()
            if candidate.is_file()
            and candidate.suffix.lower() in suffixes
        ]

        for candidate in candidate_files:
            candidate_text = normalise_match_text(
                candidate.stem
            )

            if desired_text and (
                desired_text in candidate_text
                or candidate_text in desired_text
            ):
                return candidate

            if key_tokens and all(
                token in candidate_text
                for token in key_tokens
            ):
                return candidate

    available_files = []

    for search_directory in search_directories:
        available_files.extend(
            str(candidate)
            for candidate in search_directory.iterdir()
            if candidate.is_file()
        )

    raise FileNotFoundError(
        "Could not resolve source file path.\n"
        f"Registered path: {file_path}\n"
        f"Repaired path: {repaired_path}\n"
        f"Available files checked: {available_files[:20]}"
    )


def load_file(
    file_path,
    excel_header=0,
    source_profile=None
):

    file_path = resolve_file_path(file_path)
    file_type = file_path.suffix.lower()
    source_profile = source_profile or {}

    documents = []
    tables = {}

    # Excel files
    if file_type in [".xlsx", ".xls", ".xlsm"]:

        workbook = pd.read_excel(
            file_path,
            sheet_name=None,
            header=excel_header
        )

        for sheet_name, dataframe in workbook.items():

            # Remove completely empty rows and columns
            dataframe = dataframe.dropna(
                how="all"
            )

            dataframe = dataframe.dropna(
                axis=1,
                how="all"
            )

            # Flatten multi-row column headings
            dataframe.columns = flatten_column_names(
                dataframe.columns
            )

            # Where a year column exists, remove totals,
            # footer notes and other non-year rows
            if "Año" in dataframe.columns:

                numeric_year = pd.to_numeric(
                    dataframe["Año"],
                    errors="coerce"
                )

                valid_year = numeric_year.notna()

                dataframe = dataframe.loc[
                    valid_year
                ].copy()

                dataframe["Año"] = numeric_year.loc[
                    valid_year
                ].astype(int)

            table_name = (
                f"{file_path.stem}_{sheet_name}"
            )

            tables[table_name] = dataframe

            sheet_documents = dataframe_to_documents(
                dataframe=dataframe,
                source_file=file_path,
                sheet_name=sheet_name,
                source_profile=source_profile
            )

            documents.extend(sheet_documents)

    # CSV files
    elif file_type == ".csv":

        dataframe = pd.read_csv(file_path)

        dataframe = dataframe.dropna(
            how="all"
        )

        dataframe = dataframe.dropna(
            axis=1,
            how="all"
        )

        tables[file_path.stem] = dataframe

        csv_documents = dataframe_to_documents(
            dataframe=dataframe,
            source_file=file_path,
            sheet_name="csv",
            source_profile=source_profile
        )

        documents.extend(csv_documents)

    # PDF files
    elif file_type == ".pdf":

        loader = PyPDFLoader(
            str(file_path)
        )

        documents = loader.load()

        for document in documents:

            metadata = {
                "source_file": file_path.name,
                "file_type": file_type,
                "source_type": "external_forecasting_signal",
                "record_type": "pdf_page"
            }

            metadata.update(source_profile)
            document.metadata.update(metadata)

    else:
        raise ValueError(
            f"Unsupported file type: {file_type}"
        )

    return documents, tables



# **Source Registry**

Each external dataset should be registered before loading. The registry is the control table that lets us compare sources by coverage, granularity, usefulness and limitations.

In conclusion, this section is what stops external data becoming a loose folder of files. Everything we might use for forecasting should be listed here first, even if it is only a candidate API source.

In [ ]:
SOURCE_REGISTRY_COLUMNS = [
    "source_id",
    "source_name",
    "file_path",
    "file_type",
    "domain",
    "country",
    "region",
    "granularity",
    "start_period",
    "end_period",
    "forecast_signal_category",
    "possible_target_relationship",
    "forecasting_usefulness",
    "usefulness_reason",
    "known_limitations",
    "join_keys",
    "target_model_use",
    "status"
]

source_registry = pd.DataFrame(
    [
        {
            "source_id": "cl_road_safety_annual_1972_2024",
            "source_name": "Chile road traffic collisions, fatalities, injuries, vehicle parc and population",
            "file_path": "/content/drive/MyDrive/EP/External Data/EvoluciónsiniestrostransitoChile-1972-2024-actualizado.xlsx",
            "file_type": "xlsx",
            "domain": "road_safety",
            "country": "Chile",
            "region": "national",
            "granularity": "annual",
            "start_period": "1972",
            "end_period": "2024",
            "forecast_signal_category": "collision_frequency",
            "possible_target_relationship": "collision frequency proxy",
            "forecasting_usefulness": "medium",
            "usefulness_reason": (
                "The source is directly relevant to collisions and vehicle exposure, "
                "but it is annual and national, so it is useful as lagged context "
                "rather than as a true monthly demand driver."
            ),
            "known_limitations": (
                "Annual national data; useful for long-term context, "
                "but probably too coarse for short-term spare-parts demand forecasting."
            ),
            "join_keys": "country, year",
            "target_model_use": "candidate lagged annual context feature",
            "status": "loaded"
        },
        {
            "source_id": "cl_weather_open_meteo_daily",
            "source_name": "Open-Meteo daily historical weather for Chile locations",
            "file_path": "https://archive-api.open-meteo.com/v1/archive",
            "file_type": "api",
            "domain": "weather",
            "country": "Chile",
            "region": "location",
            "granularity": "daily_to_monthly",
            "start_period": "user_selected",
            "end_period": "user_selected",
            "forecast_signal_category": "weather_collision_risk",
            "possible_target_relationship": "rain, storms, wind and temperature may affect collision frequency and repair demand",
            "forecasting_usefulness": "high",
            "usefulness_reason": (
                "The source can be materialised at daily level and aggregated to month, "
                "which matches the forecasting grain. Weather also has a plausible "
                "relationship with collision frequency."
            ),
            "known_limitations": "Requires coordinates for each forecast region; aggregate daily observations to monthly features.",
            "join_keys": "country, region, year, month",
            "target_model_use": "monthly external regressor",
            "status": "candidate_api"
        },
        {
            "source_id": "cl_public_holidays_nager_date",
            "source_name": "Chile public holidays from Nager.Date",
            "file_path": "https://date.nager.at/api/v3/PublicHolidays/{year}/CL",
            "file_type": "api",
            "domain": "calendar",
            "country": "Chile",
            "region": "national",
            "granularity": "date_to_monthly",
            "start_period": "user_selected",
            "end_period": "user_selected",
            "forecast_signal_category": "calendar_effect",
            "possible_target_relationship": "holidays and working days may affect driving patterns and repair-shop activity",
            "forecasting_usefulness": "high",
            "usefulness_reason": (
                "The source is date-level, easy to aggregate to month, and joins cleanly "
                "to monthly forecasts. It is useful for explaining working-day and "
                "holiday effects."
            ),
            "known_limitations": "National holiday calendar; local/regional exceptions may need extra handling.",
            "join_keys": "country, year, month",
            "target_model_use": "monthly calendar regressor",
            "status": "candidate_api"
        }
    ],
    columns=SOURCE_REGISTRY_COLUMNS
)

display(source_registry)

,source_id,source_name,file_path,file_type,domain,country,region,granularity,start_period,end_period,forecast_signal_category,possible_target_relationship,forecasting_usefulness,usefulness_reason,known_limitations,join_keys,target_model_use,status
0,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",/content/drive/MyDrive/EP/External Data/Evoluc...,xlsx,road_safety,Chile,national,annual,1972,2024,collision_frequency,collision frequency proxy,medium,The source is directly relevant to collisions ...,Annual national data; useful for long-term con...,"country, year",candidate lagged annual context feature,loaded
1,cl_weather_open_meteo_daily,Open-Meteo daily historical weather for Chile ...,https://archive-api.open-meteo.com/v1/archive,api,weather,Chile,location,daily_to_monthly,user_selected,user_selected,weather_collision_risk,"rain, storms, wind and temperature may affect ...",high,The source can be materialised at daily level ...,Requires coordinates for each forecast region;...,"country, region, year, month",monthly external regressor,candidate_api
2,cl_public_holidays_nager_date,Chile public holidays from Nager.Date,https://date.nager.at/api/v3/PublicHolidays/{y...,api,calendar,Chile,national,date_to_monthly,user_selected,user_selected,calendar_effect,holidays and working days may affect driving p...,high,"The source is date-level, easy to aggregate to...",National holiday calendar; local/regional exce...,"country, year, month",monthly calendar regressor,candidate_api


# **Usefulness Scoring Guide**

This is how I decide whether a source is `high`, `medium` or `low` usefulness. It is not a magic score from the model; it is a transparent assessment based on forecasting fit.

The main things I look at are:

- **Target relevance**: does the source plausibly explain collision spare-parts demand?
- **Granularity fit**: does it match monthly forecasting, or can it be safely converted to month?
- **Joinability**: does it have keys we can join on, like `country`, `region`, `year`, `month`?
- **Leakage risk**: would we know this value at the time of forecasting?
- **Coverage and reliability**: does it cover enough history and come from a stable source?

So if I get asked why a source is rated useful, the answer should come from these criteria plus the `usefulness_reason` column in the source registry.

In [ ]:
USEFULNESS_SCORING_GUIDE = pd.DataFrame(
    [
        {
            "forecasting_usefulness": "high",
            "meaning": "Strong candidate feature",
            "typical_reason": (
                "Monthly or daily source, plausible demand relationship, "
                "clean join keys, and low leakage risk."
            )
        },
        {
            "forecasting_usefulness": "medium",
            "meaning": "Useful context or candidate feature",
            "typical_reason": (
                "Relevant to the target, but too coarse, indirect, or needs "
                "careful lagging before use."
            )
        },
        {
            "forecasting_usefulness": "low",
            "meaning": "Keep for reference, but unlikely to help the model directly",
            "typical_reason": (
                "Weak target relationship, poor coverage, hard to join, "
                "or high leakage/quality risk."
            )
        }
    ]
)

display(USEFULNESS_SCORING_GUIDE)

,forecasting_usefulness,meaning,typical_reason
0,high,Strong candidate feature,"Monthly or daily source, plausible demand rela..."
1,medium,Useful context or candidate feature,"Relevant to the target, but too coarse, indire..."
2,low,"Keep for reference, but unlikely to help the m...","Weak target relationship, poor coverage, hard ..."


# **LLM-Assisted Usefulness Rating**

This optional step lets an LLM score a source against the usefulness guide, then writes the rating and reason back into the registry.

The important part is that the LLM is using our criteria, not making up a hidden score. We can still override it if domain knowledge says the rating is wrong.

In [ ]:
def build_usefulness_rating_prompt(
    source_row,
    usefulness_scoring_guide
):
    """
    Build a grounded prompt for rating one source.
    """

    source_summary = "\n".join(
        f"{column}: {source_row.get(column)}"
        for column in source_row.index
    )

    guide_text = usefulness_scoring_guide.to_string(
        index=False
    )

    return (
        "You are rating an external data source for monthly collision "
        "spare-parts forecasting.\n\n"
        "Use only this scoring guide:\n"
        f"{guide_text}\n\n"
        "Consider target relevance, granularity fit, joinability, leakage risk, "
        "coverage and reliability.\n\n"
        "Return exactly this format:\n"
        "rating: high|medium|low\n"
        "reason: one short paragraph\n\n"
        f"Source to rate:\n{source_summary}"
    )


def parse_usefulness_rating_response(
    response_text
):
    """
    Parse a simple two-field LLM response.
    """

    rating = None
    reason = None

    for line in response_text.splitlines():

        lower_line = line.lower().strip()

        if lower_line.startswith("rating:"):
            rating = line.split(":", 1)[1].strip().lower()

        if lower_line.startswith("reason:"):
            reason = line.split(":", 1)[1].strip()

    if rating not in {"high", "medium", "low"}:
        raise ValueError(
            f"Could not parse usefulness rating from response: {response_text}"
        )

    if not reason:
        raise ValueError(
            f"Could not parse usefulness reason from response: {response_text}"
        )

    return rating, reason


def rate_source_usefulness_with_llm(
    source_registry,
    source_id,
    usefulness_scoring_guide,
    model_name="gpt-4o-mini"
):
    """
    Ask an LLM to rate one source, then return the rating and reason.
    Requires OPENAI_API_KEY in the notebook environment.
    """

    from langchain_openai import ChatOpenAI

    matches = source_registry.loc[
        source_registry["source_id"] == source_id
    ]

    if matches.empty:
        raise ValueError(
            f"Source id not found in registry: {source_id}"
        )

    prompt = build_usefulness_rating_prompt(
        source_row=matches.iloc[0],
        usefulness_scoring_guide=usefulness_scoring_guide
    )

    llm = ChatOpenAI(
        model=model_name,
        temperature=0
    )

    response = llm.invoke(prompt)

    rating, reason = parse_usefulness_rating_response(
        response.content
    )

    return rating, reason, response.content


def apply_usefulness_rating(
    source_registry,
    source_id,
    rating,
    reason
):
    """
    Write a usefulness rating and reason back into the registry.
    """

    updated_registry = source_registry.copy()
    row_mask = updated_registry["source_id"] == source_id

    updated_registry.loc[
        row_mask,
        "forecasting_usefulness"
    ] = rating

    updated_registry.loc[
        row_mask,
        "usefulness_reason"
    ] = reason

    return updated_registry


# Example:
# rating, reason, raw_response = rate_source_usefulness_with_llm(
#     source_registry=source_registry,
#     source_id="cl_road_safety_annual_1972_2024",
#     usefulness_scoring_guide=USEFULNESS_SCORING_GUIDE,
#     model_name="gpt-4o-mini"
# )
#
# source_registry = apply_usefulness_rating(
#     source_registry=source_registry,
#     source_id="cl_road_safety_annual_1972_2024",
#     rating=rating,
#     reason=reason
# )
#
# display(source_registry)

In [ ]:
def source_profile_from_registry(source_row):
    """
    Convert one source registry row into metadata for LangChain Documents.
    """

    metadata_columns = [
        "source_id",
        "source_name",
        "domain",
        "country",
        "region",
        "granularity",
        "start_period",
        "end_period",
        "forecast_signal_category",
        "possible_target_relationship",
        "forecasting_usefulness",
        "usefulness_reason",
        "known_limitations",
        "join_keys",
        "target_model_use",
        "status"
    ]

    return {
        column: source_row[column]
        for column in metadata_columns
        if column in source_row.index and pd.notna(source_row[column])
    }


def load_registered_source(
    source_registry,
    source_id,
    excel_header=0
):
    """
    Load one registered source and attach registry metadata to all documents.
    """

    matches = source_registry.loc[
        source_registry["source_id"] == source_id
    ]

    if matches.empty:
        raise ValueError(
            f"Source id not found in registry: {source_id}"
        )

    source_row = matches.iloc[0]
    source_profile = source_profile_from_registry(
        source_row
    )

    documents, tables = load_file(
        file_path=source_row["file_path"],
        excel_header=excel_header,
        source_profile=source_profile
    )

    return documents, tables, source_profile

# **Load Registered Sources**

This section loads every active file-based source in the registry. API sources are kept in the registry for discovery and can be materialised later by the API feature builders.

It does **not** scan websites automatically. If a source is an API, the workbook only calls it when we run the specific API function with dates/locations. The registry and vector database are there so the LLM can explain what sources exist and how they should be used.

In [ ]:
def load_registered_sources(
    source_registry,
    default_excel_header=0,
    excel_header_by_source_id=None,
    statuses_to_load=("loaded", "active")
):
    """
    Load every active file source from the registry.

    API rows stay in the registry and are represented by registry documents,
    but they are not fetched here because they usually need date ranges,
    coordinates or credentials.
    """

    excel_header_by_source_id = excel_header_by_source_id or {}

    loaded_sources = {}
    skipped_sources = []

    for _, source_row in source_registry.iterrows():

        source_id = source_row["source_id"]
        file_type = str(source_row.get("file_type", "")).lower()
        status = source_row.get("status")

        if status not in statuses_to_load:
            skipped_sources.append(
                {
                    "source_id": source_id,
                    "reason": f"status={status}"
                }
            )
            continue

        if file_type == "api":
            skipped_sources.append(
                {
                    "source_id": source_id,
                    "reason": "api_source_loaded_by_feature_builder"
                }
            )
            continue

        excel_header = excel_header_by_source_id.get(
            source_id,
            default_excel_header
        )

        source_profile = source_profile_from_registry(
            source_row
        )

        documents, tables = load_file(
            file_path=source_row["file_path"],
            excel_header=excel_header,
            source_profile=source_profile
        )

        loaded_sources[source_id] = {
            "source_profile": source_profile,
            "file_path": source_row["file_path"],
            "documents": documents,
            "tables": tables
        }

    return loaded_sources, pd.DataFrame(skipped_sources)


excel_header_by_source_id = {
    "cl_road_safety_annual_1972_2024": [2, 3]
}

all_loaded_sources, skipped_sources = load_registered_sources(
    source_registry=source_registry,
    default_excel_header=0,
    excel_header_by_source_id=excel_header_by_source_id
)

print(f"Loaded file sources: {len(all_loaded_sources)}")
display(skipped_sources)

# Keep one selected source for quick interactive inspection.
source_id = "cl_road_safety_annual_1972_2024"
selected_source = all_loaded_sources[source_id]
documents = selected_source["documents"]
tables = selected_source["tables"]
source_profile = selected_source["source_profile"]
file_path = selected_source["file_path"]

print(f"Selected source: {source_id}")
print(f"Row-level documents created: {len(documents)}")
print(f"Tables retained: {len(tables)}")
print(f"Table names: {list(tables.keys())}")

Loaded file sources: 1


,source_id,reason
0,cl_weather_open_meteo_daily,status=candidate_api
1,cl_public_holidays_nager_date,status=candidate_api


Selected source: cl_road_safety_annual_1972_2024
Row-level documents created: 53
Tables retained: 1
Table names: ['EvoluciónsiniestrostransitoChile-1972-2024-actualizado_Chile (1972-2024)']


# **Source Documents**

This section creates the text documents used by retrieval: one summary document per table plus row-level records for period-specific lookup.

In [ ]:
def create_source_registry_documents(
    source_registry
):
    """
    Create one searchable document per registered source, including API
    candidates that are not loaded as files.
    """

    registry_documents = []

    for _, row in source_registry.iterrows():

        metadata = {
            "source_id": row.get("source_id"),
            "source_name": row.get("source_name"),
            "source_type": "external_forecasting_signal",
            "record_type": "source_registry",
            "file_type": row.get("file_type"),
            "domain": row.get("domain"),
            "country": row.get("country"),
            "region": row.get("region"),
            "granularity": row.get("granularity"),
            "forecast_signal_category": row.get("forecast_signal_category"),
            "forecasting_usefulness": row.get("forecasting_usefulness"),
            "usefulness_reason": row.get("usefulness_reason"),
            "join_keys": row.get("join_keys"),
            "target_model_use": row.get("target_model_use"),
            "status": row.get("status")
        }

        page_content = (
            f"Source id: {row.get('source_id')}\n"
            f"Source name: {row.get('source_name')}\n"
            f"Domain: {row.get('domain')}\n"
            f"Country: {row.get('country')}\n"
            f"Region: {row.get('region')}\n"
            f"Granularity: {row.get('granularity')}\n"
            f"Forecast signal category: {row.get('forecast_signal_category')}\n"
            f"Possible relationship to target: {row.get('possible_target_relationship')}\n"
            f"Forecasting usefulness: {row.get('forecasting_usefulness')}\n"
            f"Usefulness reason: {row.get('usefulness_reason')}\n"
            f"Join keys: {row.get('join_keys')}\n"
            f"Target model use: {row.get('target_model_use')}\n"
            f"Known limitations: {row.get('known_limitations')}\n"
            f"Status: {row.get('status')}"
        )

        registry_documents.append(
            Document(
                page_content=page_content,
                metadata=metadata
            )
        )

    return registry_documents


def create_table_summary_documents(
    tables,
    source_file,
    source_profile=None
):
    """
    Create one summary Document for each spreadsheet table.
    """

    source_profile = source_profile or {}
    summary_documents = []

    for table_name, dataframe in tables.items():

        column_names = ", ".join(
            str(column)
            for column in dataframe.columns
        )

        year_information = ""
        date_range = None

        if "Año" in dataframe.columns:

            years = pd.to_numeric(
                dataframe["Año"],
                errors="coerce"
            ).dropna()

            if not years.empty:
                start_year = int(years.min())
                end_year = int(years.max())
                date_range = f"{start_year}-{end_year}"
                year_information = (
                    f"\nAvailable years: "
                    f"{start_year} to "
                    f"{end_year}"
                )

        page_content = (
            f"Dataset: {Path(source_file).name}\n"
            f"Table or worksheet: {table_name}\n"
            f"Number of records: {len(dataframe)}\n"
            f"Columns: {column_names}"
            f"{year_information}\n"
            f"Forecasting role: potential external signal for collision spare-parts demand.\n"
            f"Possible relationship to target: "
            f"{source_profile.get('possible_target_relationship', 'needs assessment')}\n"
            f"Forecasting usefulness: {source_profile.get('forecasting_usefulness', 'not assessed')}\n"
            f"Usefulness reason: {source_profile.get('usefulness_reason', 'not assessed')}\n"
            f"Granularity: {source_profile.get('granularity', 'unknown')}\n"
            f"Known limitations: {source_profile.get('known_limitations', 'not yet assessed')}\n"
            f"Description: This source may provide explanatory signals or context "
            f"for monthly collision spare-parts demand forecasting."
        )

        metadata = {
            "source_file": Path(source_file).name,
            "file_type": Path(source_file).suffix.lower(),
            "source_type": "external_forecasting_signal",
            "table_name": str(table_name),
            "record_type": "dataset_summary",
            "row_count": int(len(dataframe))
        }

        metadata.update(source_profile)

        if date_range:
            metadata["date_range"] = date_range

        summary_documents.append(
            Document(
                page_content=page_content,
                metadata=metadata
            )
        )

    return summary_documents

In [ ]:
registry_documents = create_source_registry_documents(
    source_registry=source_registry
)

all_summary_documents = []
all_row_documents = []
all_vector_documents = []
all_tables = {}
all_source_profiles = {}

for loaded_source_id, loaded_source in all_loaded_sources.items():

    source_summary_documents = create_table_summary_documents(
        tables=loaded_source["tables"],
        source_file=loaded_source["file_path"],
        source_profile=loaded_source["source_profile"]
    )

    all_summary_documents.extend(
        source_summary_documents
    )
    all_row_documents.extend(
        loaded_source["documents"]
    )
    all_tables.update(
        {
            f"{loaded_source_id}__{table_name}": dataframe
            for table_name, dataframe in loaded_source["tables"].items()
        }
    )
    all_source_profiles[loaded_source_id] = loaded_source["source_profile"]

all_vector_documents = (
    registry_documents
    + all_summary_documents
    + all_row_documents
)

# Backwards-compatible aliases for the selected source.
summary_documents = create_table_summary_documents(
    tables=tables,
    source_file=file_path,
    source_profile=source_profile
)
vector_documents = summary_documents + documents

print("Registry documents:", len(registry_documents))
print("All summary documents:", len(all_summary_documents))
print("All row-level documents:", len(all_row_documents))
print("All vector documents:", len(all_vector_documents))

Registry documents: 3
All summary documents: 1
All row-level documents: 53
All vector documents: 57


# **Source Preview**

This is a human inspection step. It shows all registered sources first, including API candidates, then shows the file sources that have actually been loaded into tables.

So at this stage you should see all three sources in the registry preview, but only the road-safety workbook under loaded source tables until the weather/holiday API cells are run.

In [ ]:
print("REGISTERED SOURCES")
display(
    source_registry[
        [
            "source_id",
            "source_name",
            "file_type",
            "granularity",
            "forecasting_usefulness",
            "status"
        ]
    ]
)

print("\nSKIPPED OR API SOURCES")
display(skipped_sources)

if all_vector_documents:

    print("\nFIRST SEARCH DOCUMENT")
    print(all_vector_documents[0].page_content)

    print("\nMETADATA")
    print(all_vector_documents[0].metadata)

else:
    print("No documents were created.")

REGISTERED SOURCES


,source_id,source_name,file_type,granularity,forecasting_usefulness,status
0,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",xlsx,annual,medium,loaded
1,cl_weather_open_meteo_daily,Open-Meteo daily historical weather for Chile ...,api,daily_to_monthly,high,candidate_api
2,cl_public_holidays_nager_date,Chile public holidays from Nager.Date,api,date_to_monthly,high,candidate_api



SKIPPED OR API SOURCES


,source_id,reason
0,cl_weather_open_meteo_daily,status=candidate_api
1,cl_public_holidays_nager_date,status=candidate_api



FIRST SEARCH DOCUMENT
Source id: cl_road_safety_annual_1972_2024
Source name: Chile road traffic collisions, fatalities, injuries, vehicle parc and population
Domain: road_safety
Country: Chile
Region: national
Granularity: annual
Forecast signal category: collision_frequency
Possible relationship to target: collision frequency proxy
Forecasting usefulness: medium
Usefulness reason: The source is directly relevant to collisions and vehicle exposure, but it is annual and national, so it is useful as lagged context rather than as a true monthly demand driver.
Join keys: country, year
Target model use: candidate lagged annual context feature
Known limitations: Annual national data; useful for long-term context, but probably too coarse for short-term spare-parts demand forecasting.
Status: loaded

METADATA
{'source_id': 'cl_road_safety_annual_1972_2024', 'source_name': 'Chile road traffic collisions, fatalities, injuries, vehicle parc and population', 'source_type': 'external_forecasting_

In [ ]:
for loaded_source_id, loaded_source in all_loaded_sources.items():

    print(f"\nSOURCE: {loaded_source_id}")

    for table_name, dataframe in loaded_source["tables"].items():

        print(f"\nTABLE: {table_name}")
        print(f"Shape: {dataframe.shape}")

        display(dataframe.head())
        display(dataframe.tail())


SOURCE: cl_road_safety_annual_1972_2024

TABLE: EvoluciónsiniestrostransitoChile-1972-2024-actualizado_Chile (1972-2024)
Shape: (53, 20)


,Año,Siniestros,Fallecidos,Lesionados - Graves,Lesionados - Menos graves,Lesionados - Leves,Total lesionados,Total víctimas,Tasa motorización,Vehículos cada 100 habitantes,Parque vehicular,Población,Indicadores cada 10.000 vehículos - Siniestralidad,Indicadores cada 10.000 vehículos - Mortalidad,Indicadores cada 10.000 vehículos - Morbilidad,Indicadores cada 100.000 habitantes - Siniestralidad,Indicadores cada 100.000 habitantes - Mortalidad,Indicadores cada 100.000 habitantes - Morbilidad,Fallecidos cada 100 siniestros,Siniestros por cada fallecido
0,1972,26727.0,1792,6590.0,5624.0,11027.0,23241.0,25033.0,24.698876,4.048767,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
1,1973,23480.0,1719,6020.0,5153.0,10079.0,21252.0,22971.0,24.118185,4.146249,417767.0,10075782.0,562.035776,41.147338,508.704613,233.034022,17.060711,210.921594,7.321124,13.659104
2,1974,18356.0,1269,4935.0,3763.0,7938.0,16636.0,17905.0,23.759662,4.208814,431172.0,10244501.0,425.723377,29.431410,385.832104,179.179054,12.387133,162.389559,6.913271,14.464933
3,1975,16602.0,1054,4208.0,3479.0,7142.0,14829.0,15883.0,23.364107,4.280069,445693.0,10413219.0,372.498558,23.648565,332.717812,159.431968,10.121750,142.405533,6.348633,15.751423
4,1976,17716.0,1079,4322.0,3456.0,7355.0,15133.0,16212.0,22.670168,4.411083,466049.0,10565409.0,380.131703,23.152072,324.708346,167.679264,10.212572,143.231559,6.090540,16.418906


,Año,Siniestros,Fallecidos,Lesionados - Graves,Lesionados - Menos graves,Lesionados - Leves,Total lesionados,Total víctimas,Tasa motorización,Vehículos cada 100 habitantes,Parque vehicular,Población,Indicadores cada 10.000 vehículos - Siniestralidad,Indicadores cada 10.000 vehículos - Mortalidad,Indicadores cada 10.000 vehículos - Morbilidad,Indicadores cada 100.000 habitantes - Siniestralidad,Indicadores cada 100.000 habitantes - Mortalidad,Indicadores cada 100.000 habitantes - Morbilidad,Fallecidos cada 100 siniestros,Siniestros por cada fallecido
48,2020,64707.0,1485,6430.0,3378.0,32295.0,42103.0,43588.0,3.480201,28.733970,5591145.0,19458310.0,115.731214,2.655985,75.303001,332.541726,7.631701,216.375420,2.294960,43.573737
49,2021,80751.0,1688,8103.0,4142.0,39683.0,51928.0,53616.0,3.224718,31.010461,6102351.0,19678363.0,132.327688,2.766147,85.095072,410.354256,8.577949,263.883739,2.090377,47.838270
50,2022,86050.0,1745,8536.0,4141.0,39913.0,52590.0,54335.0,3.17163,31.529526,6251852.0,19828563.0,137.639215,2.791173,84.119074,433.969925,8.800436,265.223456,2.027891,49.312321
51,2023,78238.0,1635,7527.0,3614.0,34538.0,45679.0,47314.0,3.052718,32.757694,6538727.0,19960889.0,119.653260,2.500487,69.859164,391.956491,8.191018,228.842513,2.089777,47.851988
52,2024,75653.0,1439,7023.0,3383.0,31994.0,42400.0,43839.0,2.74701,36.403218,6727472.0,18480432.0,112.453831,2.138991,63.025160,409.368136,7.786615,229.431866,1.902106,52.573315


# **Feature Inventory**

This section checks what columns each source has before we pass anything into the forecasting layer. It shows the original source column, the model-ready feature name, whether the column is numeric, and how complete it is.

This is useful because it gives the team a quick way to see what the external data actually contains before deciding whether it should become a model feature.

In [ ]:
def create_feature_inventory(
    tables,
    source_profile,
    column_mapping=None
):
    """
    Summarise table columns so candidate model features can be reviewed.
    """

    column_mapping = column_mapping or {}
    inventory_rows = []

    for table_name, dataframe in tables.items():

        for column in dataframe.columns:

            series = dataframe[column]
            numeric_series = pd.to_numeric(
                series,
                errors="coerce"
            )

            inventory_rows.append(
                {
                    "source_id": source_profile.get("source_id"),
                    "source_name": source_profile.get("source_name"),
                    "table_name": table_name,
                    "original_column_name": column,
                    "standard_feature_name": column_mapping.get(column, column),
                    "pandas_dtype": str(series.dtype),
                    "non_null_count": int(series.notna().sum()),
                    "numeric_count": int(numeric_series.notna().sum()),
                    "min_numeric": (
                        numeric_series.min()
                        if numeric_series.notna().any()
                        else None
                    ),
                    "max_numeric": (
                        numeric_series.max()
                        if numeric_series.notna().any()
                        else None
                    ),
                    "granularity": source_profile.get("granularity"),
                    "country": source_profile.get("country"),
                    "join_keys": source_profile.get("join_keys"),
                    "forecast_signal_category": source_profile.get("forecast_signal_category"),
                    "target_model_use": source_profile.get("target_model_use")
                }
            )

    return pd.DataFrame(inventory_rows)

# **Forecasting Feature Bridge**

This section creates model-facing feature tables. Raw source tables stay untouched, but the bridge exposes stable English `snake_case` feature names.

In plain terms: this turns the external data into clean tables in a language and structure the forecasting code can reliably use. The model does not care whether the source was Spanish or English; it cares that the feature names are consistent and numeric values are usable.

In [ ]:
SPANISH_TO_ENGLISH_FEATURES = {
    "Año": "year",
    "Siniestros": "collisions",
    "Fallecidos": "fatalities",
    "Lesionados - Graves": "serious_injuries",
    "Lesionados - Menos graves": "moderate_injuries",
    "Lesionados - Leves": "minor_injuries",
    "Total lesionados": "total_injuries",
    "Total víctimas": "total_victims",
    "Tasa motorización": "motorization_rate",
    "Vehículos cada 100 habitantes": "vehicles_per_100_people",
    "Parque vehicular": "vehicle_parc",
    "Población": "population",
    "Indicadores cada 10.000 vehículos - Siniestralidad": "collisions_per_10k_vehicles",
    "Indicadores cada 10.000 vehículos - Mortalidad": "fatalities_per_10k_vehicles",
    "Indicadores cada 10.000 vehículos - Morbilidad": "injuries_per_10k_vehicles",
    "Indicadores cada 100.000 habitantes - Siniestralidad": "collisions_per_100k_population",
    "Indicadores cada 100.000 habitantes - Mortalidad": "fatalities_per_100k_population",
    "Indicadores cada 100.000 habitantes - Morbilidad": "injuries_per_100k_population",
    "Fallecidos cada 100 siniestros": "fatalities_per_100_collisions",
    "Siniestros por cada fallecido": "collisions_per_fatality"
}


def standardize_feature_columns(
    dataframe,
    column_mapping
):
    """
    Keep raw source data unchanged, but expose model-ready English feature names.
    """

    return dataframe.rename(
        columns=column_mapping
    )


def build_annual_feature_table(
    tables,
    source_profile,
    column_mapping=None
):
    """
    Build a simple modelling table for annual sources that can be exported
    or joined into existing forecasting pipelines.
    """

    column_mapping = column_mapping or {}
    feature_tables = []

    for table_name, dataframe in tables.items():

        if "Año" not in dataframe.columns:
            continue

        feature_table = standardize_feature_columns(
            dataframe.copy(),
            column_mapping
        )

        feature_table["source_id"] = source_profile.get("source_id")
        feature_table["country"] = source_profile.get("country")
        feature_table["region"] = source_profile.get("region")
        feature_table["granularity"] = source_profile.get("granularity")
        feature_table["table_name"] = table_name

        feature_tables.append(feature_table)

    if not feature_tables:
        return pd.DataFrame()

    return pd.concat(
        feature_tables,
        ignore_index=True
    )


feature_inventory_frames = []
annual_feature_frames = []

for loaded_source_id, loaded_source in all_loaded_sources.items():

    feature_inventory_frames.append(
        create_feature_inventory(
            tables=loaded_source["tables"],
            source_profile=loaded_source["source_profile"],
            column_mapping=SPANISH_TO_ENGLISH_FEATURES
        )
    )

    annual_feature_table_for_source = build_annual_feature_table(
        tables=loaded_source["tables"],
        source_profile=loaded_source["source_profile"],
        column_mapping=SPANISH_TO_ENGLISH_FEATURES
    )

    if not annual_feature_table_for_source.empty:
        annual_feature_frames.append(
            annual_feature_table_for_source
        )

feature_inventory_all_sources = (
    pd.concat(feature_inventory_frames, ignore_index=True)
    if feature_inventory_frames
    else pd.DataFrame()
)

annual_feature_table = (
    pd.concat(annual_feature_frames, ignore_index=True)
    if annual_feature_frames
    else pd.DataFrame()
)

# Backwards-compatible alias for existing checks.
feature_inventory = feature_inventory_all_sources

display(feature_inventory_all_sources)
display(annual_feature_table.head())

,source_id,source_name,table_name,original_column_name,standard_feature_name,pandas_dtype,non_null_count,numeric_count,min_numeric,max_numeric,granularity,country,join_keys,forecast_signal_category,target_model_use
0,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Año,year,int64,53,53,1.972000e+03,2.024000e+03,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
1,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Siniestros,collisions,float64,53,53,1.660200e+04,9.487900e+04,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
2,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Fallecidos,fatalities,object,53,53,1.049000e+03,1.959000e+03,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
3,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Lesionados - Graves,serious_injuries,float64,53,53,4.208000e+03,9.615000e+03,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
4,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Lesionados - Menos graves,moderate_injuries,float64,53,53,3.378000e+03,8.601000e+03,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
5,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Lesionados - Leves,minor_injuries,float64,53,53,7.142000e+03,5.038900e+04,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
6,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Total lesionados,total_injuries,float64,53,53,1.482900e+04,6.356300e+04,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
7,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Total víctimas,total_victims,float64,53,53,1.588300e+04,6.523800e+04,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
8,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Tasa motorización,motorization_rate,object,53,53,2.747010e+00,2.469888e+01,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature
9,cl_road_safety_annual_1972_2024,"Chile road traffic collisions, fatalities, inj...",EvoluciónsiniestrostransitoChile-1972-2024-act...,Vehículos cada 100 habitantes,vehicles_per_100_people,float64,53,53,4.048767e+00,3.640322e+01,annual,Chile,"country, year",collision_frequency,candidate lagged annual context feature


,year,collisions,fatalities,serious_injuries,moderate_injuries,minor_injuries,total_injuries,total_victims,motorization_rate,vehicles_per_100_people,...,collisions_per_100k_population,fatalities_per_100k_population,injuries_per_100k_population,fatalities_per_100_collisions,collisions_per_fatality,source_id,country,region,granularity,table_name
0,1972,26727.0,1792,6590.0,5624.0,11027.0,23241.0,25033.0,24.698876,4.048767,...,269.777174,18.088102,234.590164,6.704830,14.914621,cl_road_safety_annual_1972_2024,Chile,national,annual,EvoluciónsiniestrostransitoChile-1972-2024-act...
1,1973,23480.0,1719,6020.0,5153.0,10079.0,21252.0,22971.0,24.118185,4.146249,...,233.034022,17.060711,210.921594,7.321124,13.659104,cl_road_safety_annual_1972_2024,Chile,national,annual,EvoluciónsiniestrostransitoChile-1972-2024-act...
2,1974,18356.0,1269,4935.0,3763.0,7938.0,16636.0,17905.0,23.759662,4.208814,...,179.179054,12.387133,162.389559,6.913271,14.464933,cl_road_safety_annual_1972_2024,Chile,national,annual,EvoluciónsiniestrostransitoChile-1972-2024-act...
3,1975,16602.0,1054,4208.0,3479.0,7142.0,14829.0,15883.0,23.364107,4.280069,...,159.431968,10.121750,142.405533,6.348633,15.751423,cl_road_safety_annual_1972_2024,Chile,national,annual,EvoluciónsiniestrostransitoChile-1972-2024-act...
4,1976,17716.0,1079,4322.0,3456.0,7355.0,15133.0,16212.0,22.670168,4.411083,...,167.679264,10.212572,143.231559,6.090540,16.418906,cl_road_safety_annual_1972_2024,Chile,national,annual,EvoluciónsiniestrostransitoChile-1972-2024-act...


# **Monthly Forecasting Bridge**

Annual crash and vehicle-parc values are repeated across months only as slow-moving context. They must be treated as **non-additive** features: do not sum them across months, or the annual values will be inflated by 12x.

In [ ]:
def annual_context_to_monthly_features(
    annual_feature_table,
    year_column="year",
    lag_years=1
):
    """
    Convert annual source data into monthly, lagged context features.

    If lag_years=1, the 2023 annual value becomes available to all 2024
    forecast months. This avoids using full-year values that would not have
    been known at forecast time.

    Important: these repeated features are non-additive context. They should
    be joined to monthly demand rows, but not summed across months.
    """

    if annual_feature_table.empty:
        return pd.DataFrame()

    table = annual_feature_table.copy()
    table[year_column] = pd.to_numeric(
        table[year_column],
        errors="coerce"
    )
    table = table.dropna(
        subset=[year_column]
    )
    table[year_column] = table[year_column].astype(int)

    metadata_columns = {
        "source_id",
        "country",
        "region",
        "granularity",
        "table_name"
    }

    value_columns = [
        column
        for column in table.columns
        if column not in metadata_columns
        and column != year_column
    ]

    monthly_rows = []

    for _, row in table.iterrows():

        source_year = int(row[year_column])
        forecast_year = source_year + lag_years

        for month in range(1, 13):

            monthly_row = {
                "country": row.get("country"),
                "region": row.get("region"),
                "year": forecast_year,
                "month": month,
                "source_year": source_year,
                "source_id": row.get("source_id"),
                "source_granularity": row.get("granularity"),
                "feature_lag_years": lag_years,
                "feature_scope": "national",
                "feature_semantics": "repeated_annual_context",
                "aggregation_rule": "first_or_mean_not_sum"
            }

            for column in value_columns:
                monthly_row[f"lag_{lag_years}yr_national_annual_{column}_context"] = row[column]

            monthly_rows.append(monthly_row)

    return pd.DataFrame(monthly_rows)


def repeat_national_features_to_regions(
    monthly_features,
    forecast_regions,
    scope_label="national_repeated_to_region"
):
    """
    Repeat national monthly context across regions for optional regional tests.

    These rows are labelled clearly because the values are not truly regional.
    """

    if monthly_features.empty:
        return monthly_features.copy()

    if forecast_regions.empty:
        return monthly_features.copy()

    features = monthly_features.copy()
    region_columns_to_drop = [
        column
        for column in ["country", "region"]
        if column in features.columns
    ]
    features = features.drop(
        columns=region_columns_to_drop
    )

    repeated = features.merge(
        forecast_regions,
        how="cross"
    )

    ordered_columns = [
        column
        for column in ["country", "region", "year", "month"]
        if column in repeated.columns
    ]
    remaining_columns = [
        column
        for column in repeated.columns
        if column not in ordered_columns
    ]
    repeated = repeated[ordered_columns + remaining_columns]
    repeated["feature_scope"] = scope_label

    return repeated


def collapse_repeated_regional_features_to_national(
    monthly_features,
    national_region_name="national"
):
    """
    Convert an older regional cache back to national month grain by taking the
    first repeated value per month. This is for national-only calendar/context
    features, not for true regional weather.
    """

    if monthly_features.empty:
        return monthly_features.copy()

    if "region" not in monthly_features.columns:
        return monthly_features.copy()

    if set(monthly_features["region"].dropna().unique()) == {national_region_name}:
        return monthly_features.copy()

    collapsed = (
        monthly_features
        .sort_values(["year", "month"])
        .groupby(["year", "month"], as_index=False)
        .first()
    )
    collapsed["region"] = national_region_name

    return collapsed


DEFAULT_FORECAST_REGION_NAMES = [
    "Antofagasta",
    "Arica",
    "Concepcion",
    "La Serena",
    "Puerto Montt",
    "Punta Arenas",
    "Santiago",
    "Temuco",
    "Valparaiso"
]

forecast_regions = pd.DataFrame(
    {
        "region": DEFAULT_FORECAST_REGION_NAMES
    }
).assign(country="Chile")

monthly_annual_context_features = annual_context_to_monthly_features(
    annual_feature_table=annual_feature_table,
    year_column="year",
    lag_years=1
)

display(monthly_annual_context_features.head(15))


,country,region,year,month,source_year,source_id,source_granularity,feature_lag_years,feature_scope,feature_semantics,...,lag_1yr_national_annual_vehicle_parc_context,lag_1yr_national_annual_population_context,lag_1yr_national_annual_collisions_per_10k_vehicles_context,lag_1yr_national_annual_fatalities_per_10k_vehicles_context,lag_1yr_national_annual_injuries_per_10k_vehicles_context,lag_1yr_national_annual_collisions_per_100k_population_context,lag_1yr_national_annual_fatalities_per_100k_population_context,lag_1yr_national_annual_injuries_per_100k_population_context,lag_1yr_national_annual_fatalities_per_100_collisions_context,lag_1yr_national_annual_collisions_per_fatality_context
0,Chile,national,1973,1,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
1,Chile,national,1973,2,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
2,Chile,national,1973,3,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
3,Chile,national,1973,4,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
4,Chile,national,1973,5,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
5,Chile,national,1973,6,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
6,Chile,national,1973,7,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
7,Chile,national,1973,8,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
8,Chile,national,1973,9,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621
9,Chile,national,1973,10,1972,cl_road_safety_annual_1972_2024,annual,1,national,repeated_annual_context,...,401114.0,9907065.0,666.319301,44.675579,579.411339,269.777174,18.088102,234.590164,6.704830,14.914621


# **Monthly API Feature Sources**

For national Chile forecasts, weather should not use a single city unless the demand model is Santiago-only. Use a region/location list, aggregate each location to monthly features, then combine them into national or regional weather features.

> **Weather weighting note:** the Chile weather locations start with equal weights (`1.0`) because we do not yet know whether the team's demand should be weighted by population, vehicle parc, collision share, or internal sales/demand share. The `weight` column is kept so this can be tested later without changing the pipeline structure.


In [ ]:
FETCH_LIVE_API_FEATURES = True
API_START_DATE = "2021-01-01"
API_END_DATE = "2026-04-30"
API_START_YEAR = int(API_START_DATE[:4])
API_END_YEAR = int(API_END_DATE[:4])

API_REQUEST_RETRIES = 3
API_REQUEST_PAUSE_SECONDS = 1.0
API_CACHE_DIRECTORY = Path(
    "/content/drive/MyDrive/EP/External Data/processed/api_cache"
)

WEATHER_CACHE_PATH = (
    API_CACHE_DIRECTORY
    / f"monthly_weather_features_{API_START_DATE}_{API_END_DATE}.csv"
)
WEATHER_LOCATION_CACHE_PATH = (
    API_CACHE_DIRECTORY
    / f"monthly_regional_weather_features_{API_START_DATE}_{API_END_DATE}.csv"
)

CHILE_WEATHER_LOCATIONS = pd.DataFrame(
    [
        {"region": "Arica", "latitude": -18.4783, "longitude": -70.3126, "weight": 1.0},
        {"region": "Antofagasta", "latitude": -23.6509, "longitude": -70.3975, "weight": 1.0},
        {"region": "La Serena", "latitude": -29.9027, "longitude": -71.2519, "weight": 1.0},
        {"region": "Valparaiso", "latitude": -33.0472, "longitude": -71.6127, "weight": 1.0},
        {"region": "Santiago", "latitude": -33.4489, "longitude": -70.6693, "weight": 1.0},
        {"region": "Concepcion", "latitude": -36.8201, "longitude": -73.0444, "weight": 1.0},
        {"region": "Temuco", "latitude": -38.7359, "longitude": -72.5904, "weight": 1.0},
        {"region": "Puerto Montt", "latitude": -41.4693, "longitude": -72.9424, "weight": 1.0},
        {"region": "Punta Arenas", "latitude": -53.1638, "longitude": -70.9171, "weight": 1.0}
    ]
)


def robust_get_json(
    url,
    params=None,
    timeout=60,
    retries=API_REQUEST_RETRIES,
    pause_seconds=API_REQUEST_PAUSE_SECONDS
):
    """
    Request JSON with simple retries so temporary API limits/timeouts do not
    crash the workbook immediately.
    """

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout
            )
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            last_error = error
            if attempt < retries:
                time.sleep(pause_seconds * attempt)

    raise last_error


def save_api_cache(data, path):
    """
    Save a materialised API output so the notebook can be rerun without
    repeatedly calling the live API.
    """

    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )
    data.to_csv(
        path,
        index=False
    )


def load_api_cache(path):
    """
    Load a cached API output if it exists.
    """

    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    return pd.read_csv(path)


def fetch_open_meteo_daily_weather(
    latitude,
    longitude,
    start_date,
    end_date,
    timezone="America/Santiago"
):
    """
    Fetch daily weather data from Open-Meteo's historical archive API.
    """

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "rain_sum",
            "snowfall_sum",
            "precipitation_hours",
            "wind_speed_10m_max",
            "wind_gusts_10m_max"
        ],
        "timezone": timezone
    }

    daily = robust_get_json(
        url=url,
        params=params
    )["daily"]

    weather = pd.DataFrame(daily)
    weather["date"] = pd.to_datetime(weather["time"])
    weather = weather.drop(
        columns=["time"]
    )

    return weather


def aggregate_weather_to_monthly(
    daily_weather,
    country="Chile",
    region="Santiago",
    weight=1.0
):
    """
    Convert daily weather observations into monthly forecasting features.
    """

    weather = daily_weather.copy()
    weather["year"] = weather["date"].dt.year
    weather["month"] = weather["date"].dt.month
    weather["is_rain_day"] = weather["rain_sum"].fillna(0) > 0

    monthly_weather = (
        weather
        .groupby(["year", "month"], as_index=False)
        .agg(
            avg_temperature=("temperature_2m_mean", "mean"),
            max_temperature=("temperature_2m_max", "max"),
            min_temperature=("temperature_2m_min", "min"),
            precipitation_sum=("precipitation_sum", "sum"),
            rain_sum=("rain_sum", "sum"),
            snowfall_sum=("snowfall_sum", "sum"),
            precipitation_hours=("precipitation_hours", "sum"),
            max_wind_speed=("wind_speed_10m_max", "max"),
            max_wind_gust=("wind_gusts_10m_max", "max"),
            rain_days=("is_rain_day", "sum"),
            weather_days=("date", "count")
        )
    )

    monthly_weather["country"] = country
    monthly_weather["region"] = region
    monthly_weather["weather_location_weight"] = weight
    monthly_weather["source_id"] = "cl_weather_open_meteo_daily"

    return monthly_weather


def fetch_chile_weather_locations(
    weather_locations,
    start_date,
    end_date,
    country="Chile"
):
    """
    Fetch and aggregate weather for multiple Chile locations.

    If one location fails after retries, the notebook records the failure and
    carries on with the remaining locations.
    """

    monthly_location_frames = []
    api_failures = []

    for _, location in weather_locations.iterrows():
        try:
            daily_weather = fetch_open_meteo_daily_weather(
                latitude=location["latitude"],
                longitude=location["longitude"],
                start_date=start_date,
                end_date=end_date
            )

            monthly_location_weather = aggregate_weather_to_monthly(
                daily_weather=daily_weather,
                country=country,
                region=location["region"],
                weight=location.get("weight", 1.0)
            )

            monthly_location_frames.append(
                monthly_location_weather
            )
            time.sleep(API_REQUEST_PAUSE_SECONDS)

        except Exception as error:
            api_failures.append(
                {
                    "source_id": "cl_weather_open_meteo_daily",
                    "region": location["region"],
                    "error": str(error)
                }
            )

    if not monthly_location_frames:
        return pd.DataFrame(), pd.DataFrame(api_failures)

    return (
        pd.concat(
            monthly_location_frames,
            ignore_index=True
        ),
        pd.DataFrame(api_failures)
    )


def aggregate_location_weather_to_national(
    monthly_location_weather,
    country="Chile",
    national_region_name="national"
):
    """
    Combine location-level monthly weather into national monthly features.

    This averages conditions across locations. It does not sum rainfall across
    locations, because that would scale with the number of weather points.
    """

    if monthly_location_weather.empty:
        return pd.DataFrame()

    weather = monthly_location_weather.copy()

    national_weather = (
        weather
        .groupby(["year", "month"], as_index=False)
        .agg(
            avg_temperature=("avg_temperature", "mean"),
            max_temperature=("max_temperature", "max"),
            min_temperature=("min_temperature", "min"),
            avg_precipitation_sum=("precipitation_sum", "mean"),
            avg_rain_sum=("rain_sum", "mean"),
            avg_snowfall_sum=("snowfall_sum", "mean"),
            avg_precipitation_hours=("precipitation_hours", "mean"),
            max_wind_speed=("max_wind_speed", "max"),
            max_wind_gust=("max_wind_gust", "max"),
            avg_rain_days=("rain_days", "mean"),
            weather_locations=("region", "nunique")
        )
    )

    national_weather["country"] = country
    national_weather["region"] = national_region_name
    national_weather["source_id"] = "cl_weather_open_meteo_daily"
    national_weather["feature_scope"] = "national_weather_context"
    national_weather["feature_semantics"] = "national_weather_context"
    national_weather["aggregation_rule"] = "location_average_not_sum"

    return national_weather


weather_api_status = []
monthly_regional_weather_features = pd.DataFrame()
monthly_weather_features = pd.DataFrame()

if FETCH_LIVE_API_FEATURES:
    try:
        monthly_regional_weather_features, weather_api_failures = fetch_chile_weather_locations(
            weather_locations=CHILE_WEATHER_LOCATIONS,
            start_date=API_START_DATE,
            end_date=API_END_DATE
        )
        monthly_regional_weather_features["feature_scope"] = "regional_weather_proxy"
        monthly_regional_weather_features["feature_semantics"] = "regional_weather_context"
        monthly_regional_weather_features["aggregation_rule"] = "daily_weather_to_monthly_by_location"

        monthly_weather_features = aggregate_location_weather_to_national(
            monthly_location_weather=monthly_regional_weather_features,
            country="Chile",
            national_region_name="national"
        )

        if not monthly_weather_features.empty:
            save_api_cache(monthly_regional_weather_features, WEATHER_LOCATION_CACHE_PATH)
            save_api_cache(monthly_weather_features, WEATHER_CACHE_PATH)
            weather_api_status.append(
                {
                    "source_id": "cl_weather_open_meteo_daily",
                    "status": "live_api_refreshed",
                    "national_rows": len(monthly_weather_features),
                    "regional_rows": len(monthly_regional_weather_features),
                    "cache_path": str(WEATHER_CACHE_PATH)
                }
            )
        else:
            weather_api_status.append(
                {
                    "source_id": "cl_weather_open_meteo_daily",
                    "status": "live_api_returned_no_rows",
                    "national_rows": 0,
                    "regional_rows": len(monthly_regional_weather_features),
                    "cache_path": str(WEATHER_CACHE_PATH)
                }
            )

    except Exception as error:
        weather_api_failures = pd.DataFrame(
            [
                {
                    "source_id": "cl_weather_open_meteo_daily",
                    "region": "all",
                    "error": str(error)
                }
            ]
        )
        monthly_weather_features = load_api_cache(WEATHER_CACHE_PATH)
        monthly_regional_weather_features = load_api_cache(WEATHER_LOCATION_CACHE_PATH)

        if monthly_weather_features.empty and not monthly_regional_weather_features.empty:
            monthly_weather_features = aggregate_location_weather_to_national(
                monthly_location_weather=monthly_regional_weather_features,
                country="Chile",
                national_region_name="national"
            )

        weather_api_status.append(
            {
                "source_id": "cl_weather_open_meteo_daily",
                "status": "live_api_failed_loaded_cache"
                if not monthly_weather_features.empty
                else "live_api_failed_no_cache",
                "national_rows": len(monthly_weather_features),
                "regional_rows": len(monthly_regional_weather_features),
                "cache_path": str(WEATHER_CACHE_PATH)
            }
        )
else:
    weather_api_failures = pd.DataFrame()
    monthly_weather_features = load_api_cache(WEATHER_CACHE_PATH)
    monthly_regional_weather_features = load_api_cache(WEATHER_LOCATION_CACHE_PATH)

    if monthly_weather_features.empty and not monthly_regional_weather_features.empty:
        monthly_weather_features = aggregate_location_weather_to_national(
            monthly_location_weather=monthly_regional_weather_features,
            country="Chile",
            national_region_name="national"
        )

    weather_api_status.append(
        {
            "source_id": "cl_weather_open_meteo_daily",
            "status": "loaded_cache_live_api_off"
            if not monthly_weather_features.empty
            else "live_api_off_no_cache",
            "national_rows": len(monthly_weather_features),
            "regional_rows": len(monthly_regional_weather_features),
            "cache_path": str(WEATHER_CACHE_PATH)
        }
    )


if (
    not monthly_weather_features.empty
    and "region" in monthly_weather_features.columns
    and set(monthly_weather_features["region"].dropna().unique()) != {"national"}
):
    if monthly_regional_weather_features.empty:
        monthly_regional_weather_features = monthly_weather_features.copy()

    monthly_weather_features = aggregate_location_weather_to_national(
        monthly_location_weather=monthly_regional_weather_features,
        country="Chile",
        national_region_name="national"
    )
display(pd.DataFrame(weather_api_status))

if not weather_api_failures.empty:
    display(weather_api_failures)

display(monthly_weather_features.head())
display(monthly_regional_weather_features.head())



,source_id,status,national_rows,regional_rows,cache_path
0,cl_weather_open_meteo_daily,live_api_refreshed,64,448,/content/drive/MyDrive/EP/External Data/proces...


,source_id,region,error
0,cl_weather_open_meteo_daily,Concepcion,429 Client Error: Too Many Requests for url: h...
1,cl_weather_open_meteo_daily,Temuco,429 Client Error: Too Many Requests for url: h...


,year,month,avg_temperature,max_temperature,min_temperature,avg_precipitation_sum,avg_rain_sum,avg_snowfall_sum,avg_precipitation_hours,max_wind_speed,max_wind_gust,avg_rain_days,weather_locations,country,region,source_id,feature_scope,feature_semantics,aggregation_rule
0,2021,1,17.428111,31.3,3.6,33.285714,33.271429,0.01,61.285714,56.5,110.9,7.285714,7,Chile,national,cl_weather_open_meteo_daily,national_weather_context,national_weather_context,location_average_not_sum
1,2021,2,17.769388,31.5,4.5,10.000000,10.000000,0.00,29.142857,59.8,115.6,5.142857,7,Chile,national,cl_weather_open_meteo_daily,national_weather_context,national_weather_context,location_average_not_sum
2,2021,3,16.879724,32.5,2.5,22.771429,22.771429,0.00,50.714286,66.3,126.7,6.857143,7,Chile,national,cl_weather_open_meteo_daily,national_weather_context,national_weather_context,location_average_not_sum
3,2021,4,14.937619,29.3,1.8,43.900000,43.900000,0.00,71.857143,52.5,101.2,8.142857,7,Chile,national,cl_weather_open_meteo_daily,national_weather_context,national_weather_context,location_average_not_sum
4,2021,5,13.303226,24.6,-1.5,37.542857,37.500000,0.04,73.142857,46.4,90.7,8.714286,7,Chile,national,cl_weather_open_meteo_daily,national_weather_context,national_weather_context,location_average_not_sum


,year,month,avg_temperature,max_temperature,min_temperature,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,max_wind_speed,max_wind_gust,rain_days,weather_days,country,region,weather_location_weight,source_id,feature_scope,feature_semantics,aggregation_rule
0,2021,1,22.400000,27.2,17.9,0.5,0.5,0.0,4.0,20.3,41.4,2,31,Chile,Arica,1.0,cl_weather_open_meteo_daily,regional_weather_proxy,regional_weather_context,daily_weather_to_monthly_by_location
1,2021,2,22.742857,29.5,18.3,0.2,0.2,0.0,2.0,19.6,40.0,1,28,Chile,Arica,1.0,cl_weather_open_meteo_daily,regional_weather_proxy,regional_weather_context,daily_weather_to_monthly_by_location
2,2021,3,22.358065,28.4,17.6,0.4,0.4,0.0,2.0,18.4,37.4,1,31,Chile,Arica,1.0,cl_weather_open_meteo_daily,regional_weather_proxy,regional_weather_context,daily_weather_to_monthly_by_location
3,2021,4,19.763333,24.9,15.7,0.3,0.3,0.0,3.0,17.6,35.6,1,30,Chile,Arica,1.0,cl_weather_open_meteo_daily,regional_weather_proxy,regional_weather_context,daily_weather_to_monthly_by_location
4,2021,5,18.219355,22.8,14.4,0.0,0.0,0.0,0.0,17.0,35.3,0,31,Chile,Arica,1.0,cl_weather_open_meteo_daily,regional_weather_proxy,regional_weather_context,daily_weather_to_monthly_by_location


In [ ]:
CALENDAR_CACHE_PATH = (
    API_CACHE_DIRECTORY
    / f"monthly_calendar_features_{API_START_DATE}_{API_END_DATE}.csv"
)


def fetch_public_holidays(
    years,
    country_code="CL"
):
    """
    Fetch public holidays from Nager.Date and return date-level records.

    If a single year fails, the notebook records the failure and carries on.
    """

    holiday_frames = []
    api_failures = []

    for year in years:

        url = (
            "https://date.nager.at/api/v3/"
            f"PublicHolidays/{year}/{country_code}"
        )

        try:
            holidays = pd.DataFrame(
                robust_get_json(url=url)
            )
            time.sleep(API_REQUEST_PAUSE_SECONDS)
        except Exception as error:
            api_failures.append(
                {
                    "source_id": "cl_public_holidays_nager_date",
                    "year": year,
                    "error": str(error)
                }
            )
            continue

        if holidays.empty:
            continue

        holidays["date"] = pd.to_datetime(
            holidays["date"]
        )
        holidays["year"] = holidays["date"].dt.year
        holidays["month"] = holidays["date"].dt.month

        holiday_frames.append(holidays)

    if not holiday_frames:
        return pd.DataFrame(), pd.DataFrame(api_failures)

    return (
        pd.concat(
            holiday_frames,
            ignore_index=True
        ),
        pd.DataFrame(api_failures)
    )


def build_monthly_calendar_features(
    start_year,
    end_year,
    country="Chile",
    country_code="CL"
):
    """
    Build monthly public holiday and working-day features.
    """

    calendar = pd.DataFrame(
        {
            "date": pd.date_range(
                start=f"{start_year}-01-01",
                end=f"{end_year}-12-31",
                freq="D"
            )
        }
    )
    calendar["year"] = calendar["date"].dt.year
    calendar["month"] = calendar["date"].dt.month
    calendar["is_weekend"] = calendar["date"].dt.dayofweek >= 5

    holidays, holiday_api_failures = fetch_public_holidays(
        years=range(start_year, end_year + 1),
        country_code=country_code
    )

    holiday_dates = set(
        holidays["date"].dt.date
    ) if not holidays.empty else set()

    calendar["is_public_holiday"] = calendar["date"].dt.date.isin(
        holiday_dates
    )
    calendar["is_working_day"] = (
        ~calendar["is_weekend"]
        & ~calendar["is_public_holiday"]
    )

    monthly_calendar = (
        calendar
        .groupby(["year", "month"], as_index=False)
        .agg(
            days_in_month=("date", "count"),
            weekend_days=("is_weekend", "sum"),
            public_holidays=("is_public_holiday", "sum"),
            working_days=("is_working_day", "sum")
        )
    )

    monthly_calendar["country"] = country
    monthly_calendar["region"] = "national"
    monthly_calendar["source_id"] = "cl_public_holidays_nager_date"
    monthly_calendar["feature_scope"] = "national_calendar_context"
    monthly_calendar["feature_semantics"] = "calendar_context"
    monthly_calendar["aggregation_rule"] = "monthly_calendar_counts"

    return monthly_calendar, holiday_api_failures


calendar_api_status = []
monthly_calendar_features = pd.DataFrame()

if FETCH_LIVE_API_FEATURES:
    try:
        monthly_calendar_features, holiday_api_failures = build_monthly_calendar_features(
            start_year=API_START_YEAR,
            end_year=API_END_YEAR,
            country="Chile",
            country_code="CL"
        )

        if not monthly_calendar_features.empty:
            save_api_cache(monthly_calendar_features, CALENDAR_CACHE_PATH)
            calendar_api_status.append(
                {
                    "source_id": "cl_public_holidays_nager_date",
                    "status": "live_api_refreshed",
                    "rows": len(monthly_calendar_features),
                    "cache_path": str(CALENDAR_CACHE_PATH)
                }
            )
        else:
            calendar_api_status.append(
                {
                    "source_id": "cl_public_holidays_nager_date",
                    "status": "live_api_returned_no_rows",
                    "rows": 0,
                    "cache_path": str(CALENDAR_CACHE_PATH)
                }
            )

    except Exception as error:
        holiday_api_failures = pd.DataFrame(
            [
                {
                    "source_id": "cl_public_holidays_nager_date",
                    "year": "all",
                    "error": str(error)
                }
            ]
        )
        monthly_calendar_features = collapse_repeated_regional_features_to_national(
            load_api_cache(CALENDAR_CACHE_PATH)
        )
        calendar_api_status.append(
            {
                "source_id": "cl_public_holidays_nager_date",
                "status": "live_api_failed_loaded_cache"
                if not monthly_calendar_features.empty
                else "live_api_failed_no_cache",
                "rows": len(monthly_calendar_features),
                "cache_path": str(CALENDAR_CACHE_PATH)
            }
        )
else:
    holiday_api_failures = pd.DataFrame()
    monthly_calendar_features = collapse_repeated_regional_features_to_national(
        load_api_cache(CALENDAR_CACHE_PATH)
    )
    calendar_api_status.append(
        {
            "source_id": "cl_public_holidays_nager_date",
            "status": "loaded_cache_live_api_off"
            if not monthly_calendar_features.empty
            else "live_api_off_no_cache",
            "rows": len(monthly_calendar_features),
            "cache_path": str(CALENDAR_CACHE_PATH)
        }
    )

monthly_regional_calendar_features = repeat_national_features_to_regions(
    monthly_features=monthly_calendar_features,
    forecast_regions=forecast_regions,
    scope_label="national_calendar_repeated_to_region"
)

display(pd.DataFrame(calendar_api_status))

if not holiday_api_failures.empty:
    display(holiday_api_failures)

display(monthly_calendar_features.head())
display(monthly_regional_calendar_features.head())


,source_id,status,rows,cache_path
0,cl_public_holidays_nager_date,live_api_refreshed,72,/content/drive/MyDrive/EP/External Data/proces...


,year,month,days_in_month,weekend_days,public_holidays,working_days,country,region,source_id,feature_scope,feature_semantics,aggregation_rule
0,2021,1,31,10,1,20,Chile,national,cl_public_holidays_nager_date,national_calendar_context,calendar_context,monthly_calendar_counts
1,2021,2,28,8,0,20,Chile,national,cl_public_holidays_nager_date,national_calendar_context,calendar_context,monthly_calendar_counts
2,2021,3,31,8,0,23,Chile,national,cl_public_holidays_nager_date,national_calendar_context,calendar_context,monthly_calendar_counts
3,2021,4,30,8,2,21,Chile,national,cl_public_holidays_nager_date,national_calendar_context,calendar_context,monthly_calendar_counts
4,2021,5,31,10,2,20,Chile,national,cl_public_holidays_nager_date,national_calendar_context,calendar_context,monthly_calendar_counts


,country,region,year,month,days_in_month,weekend_days,public_holidays,working_days,source_id,feature_scope,feature_semantics,aggregation_rule
0,Chile,Antofagasta,2021,1,31,10,1,20,cl_public_holidays_nager_date,national_calendar_repeated_to_region,calendar_context,monthly_calendar_counts
1,Chile,Arica,2021,1,31,10,1,20,cl_public_holidays_nager_date,national_calendar_repeated_to_region,calendar_context,monthly_calendar_counts
2,Chile,Concepcion,2021,1,31,10,1,20,cl_public_holidays_nager_date,national_calendar_repeated_to_region,calendar_context,monthly_calendar_counts
3,Chile,La Serena,2021,1,31,10,1,20,cl_public_holidays_nager_date,national_calendar_repeated_to_region,calendar_context,monthly_calendar_counts
4,Chile,Puerto Montt,2021,1,31,10,1,20,cl_public_holidays_nager_date,national_calendar_repeated_to_region,calendar_context,monthly_calendar_counts


In [ ]:
MONTHLY_JOIN_KEYS = ["year", "month"]
REGIONAL_MONTHLY_JOIN_KEYS = ["region", "year", "month"]

MODEL_HANDOFF_METADATA_COLUMNS = [
    "country",
    "source_id",
    "source_year",
    "source_granularity",
    "feature_lag_years",
    "feature_scope",
    "feature_semantics",
    "aggregation_rule",
    "weather_location_weight"
]


def prepare_monthly_feature_table_for_handoff(
    feature_table,
    join_keys=MONTHLY_JOIN_KEYS
):
    """
    Keep model handoff tables focused on join keys and feature values.

    `region` is kept only for optional regional exports.
    """

    metadata_columns = MODEL_HANDOFF_METADATA_COLUMNS.copy()

    if "region" not in join_keys:
        metadata_columns.append("region")

    columns_to_drop = [
        column
        for column in metadata_columns
        if column in feature_table.columns
    ]

    return feature_table.drop(
        columns=columns_to_drop
    )


def combine_monthly_external_features(
    feature_tables,
    join_keys=MONTHLY_JOIN_KEYS
):
    """
    Combine monthly external features on the requested forecasting keys.
    """

    non_empty_tables = [
        prepare_monthly_feature_table_for_handoff(
            table.copy(),
            join_keys=join_keys
        )
        for table in feature_tables
        if table is not None and not table.empty
    ]

    if not non_empty_tables:
        return pd.DataFrame()

    combined = non_empty_tables[0]

    for table in non_empty_tables[1:]:

        available_join_keys = [
            key
            for key in join_keys
            if key in combined.columns and key in table.columns
        ]

        if available_join_keys != join_keys:
            raise ValueError(
                "Monthly feature tables must share the requested join keys: "
                f"{join_keys}. Available keys were: {available_join_keys}"
            )

        combined = combined.merge(
            table,
            on=join_keys,
            how="outer",
            suffixes=("", "_duplicate")
        )

        duplicate_columns = [
            column
            for column in combined.columns
            if column.endswith("_duplicate")
        ]

        combined = combined.drop(
            columns=duplicate_columns
        )

    return combined


# Default handoff merge for the current forecasting model:
# sales = sales.merge(
#     monthly_external_features,
#     on="date",
#     how="left"
# )

# Optional regional experiment:
# regional_model_data = regional_demand_data.merge(
#     monthly_regional_external_features,
#     on=["region", "year", "month"],
#     how="left"
# )



# **Monthly Feature Assembly**

This section creates the structured model handoff tables.

The default table is `monthly_external_features`, which is national monthly grain and should match the team's current monthly forecasting data. It joins on `year` and `month`.

The optional table is `monthly_regional_external_features`, which keeps regional weather proxy rows for later experiments. It joins on `region`, `year` and `month`.

The split is important:

- `monthly_external_features` is the safe default for the forecasting models.
- `monthly_regional_external_features` is optional testing material if the team later wants regional modelling.
- the vector database is for the LLM/search layer, so people can ask questions about the external data.


In [ ]:
# Build the current monthly feature handoff from all available materialised sources.

MODEL_START_DATE = API_START_DATE
MODEL_END_DATE = API_END_DATE

monthly_feature_tables = [
    monthly_annual_context_features,
    monthly_weather_features,
    monthly_calendar_features
]

monthly_external_features_unfiltered = combine_monthly_external_features(
    monthly_feature_tables,
    join_keys=MONTHLY_JOIN_KEYS
)

monthly_regional_annual_context_features = repeat_national_features_to_regions(
    monthly_features=monthly_annual_context_features,
    forecast_regions=forecast_regions,
    scope_label="national_annual_context_repeated_to_region"
)

monthly_regional_feature_tables = [
    monthly_regional_annual_context_features,
    monthly_regional_weather_features,
    monthly_regional_calendar_features
]

monthly_regional_external_features_unfiltered = combine_monthly_external_features(
    monthly_regional_feature_tables,
    join_keys=REGIONAL_MONTHLY_JOIN_KEYS
)


def filter_monthly_features_to_model_window(
    monthly_features,
    start_date,
    end_date
):
    """
    Keep the model handoff aligned to the current forecasting window.

    Older source history is still available in source previews, source documents,
    and the vector database, but the exported feature table should only contain
    rows that can be merged into the team's monthly model data.
    """

    if monthly_features.empty:
        return monthly_features.copy()

    filtered = monthly_features.copy()
    feature_month = pd.to_datetime(
        filtered["year"].astype(int).astype(str)
        + "-"
        + filtered["month"].astype(int).astype(str).str.zfill(2)
        + "-01"
    )

    start_month = pd.Timestamp(start_date).to_period("M").to_timestamp()
    end_month = pd.Timestamp(end_date).to_period("M").to_timestamp()

    return filtered.loc[
        (feature_month >= start_month)
        & (feature_month <= end_month)
    ].copy()




def add_month_start_date(
    monthly_features,
    date_column="date"
):
    """
    Add a month-start date so downstream notebooks can merge on `date` directly.

    `year` and `month` are kept as audit-friendly columns, but `date` is the
    simplest join key when the sales notebook already works from a monthly date.
    """

    if monthly_features.empty:
        return monthly_features.copy()

    features = monthly_features.copy()
    features[date_column] = pd.to_datetime(
        features["year"].astype(int).astype(str)
        + "-"
        + features["month"].astype(int).astype(str).str.zfill(2)
        + "-01"
    )

    front_columns = [
        column
        for column in [date_column, "year", "month", "region"]
        if column in features.columns
    ]
    remaining_columns = [
        column
        for column in features.columns
        if column not in front_columns
    ]

    return features[front_columns + remaining_columns]

monthly_external_features = add_month_start_date(
    filter_monthly_features_to_model_window(
        monthly_features=monthly_external_features_unfiltered,
        start_date=MODEL_START_DATE,
        end_date=MODEL_END_DATE
    )
)

monthly_regional_external_features = add_month_start_date(
    filter_monthly_features_to_model_window(
        monthly_features=monthly_regional_external_features_unfiltered,
        start_date=MODEL_START_DATE,
        end_date=MODEL_END_DATE
    )
)

regional_feature_coverage = (
    monthly_regional_external_features
    .groupby("region", as_index=False)
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        row_count=("month", "count")
    )
    if not monthly_regional_external_features.empty
    and "region" in monthly_regional_external_features.columns
    else pd.DataFrame()
)

monthly_feature_window = pd.DataFrame(
    [
        {
            "handoff_table": "monthly_external_features",
            "model_start_date": MODEL_START_DATE,
            "model_end_date": MODEL_END_DATE,
            "grain": "national_month",
            "join_keys": MONTHLY_JOIN_KEYS,
            "rows_before_model_window_filter": len(monthly_external_features_unfiltered),
            "rows_after_model_window_filter": len(monthly_external_features),
            "regions_in_export": 0,
            "earliest_export_year_month": (
                monthly_external_features[["year", "month"]]
                .assign(month=lambda data: data["month"].astype(int).astype(str).str.zfill(2))
                .assign(year_month=lambda data: data["year"].astype(int).astype(str) + "-" + data["month"])
                ["year_month"]
                .min()
                if not monthly_external_features.empty
                else pd.NA
            ),
            "latest_export_year_month": (
                monthly_external_features[["year", "month"]]
                .assign(month=lambda data: data["month"].astype(int).astype(str).str.zfill(2))
                .assign(year_month=lambda data: data["year"].astype(int).astype(str) + "-" + data["month"])
                ["year_month"]
                .max()
                if not monthly_external_features.empty
                else pd.NA
            )
        },
        {
            "handoff_table": "monthly_regional_external_features",
            "model_start_date": MODEL_START_DATE,
            "model_end_date": MODEL_END_DATE,
            "grain": "region_month_optional",
            "join_keys": REGIONAL_MONTHLY_JOIN_KEYS,
            "rows_before_model_window_filter": len(monthly_regional_external_features_unfiltered),
            "rows_after_model_window_filter": len(monthly_regional_external_features),
            "regions_in_export": (
                monthly_regional_external_features["region"].nunique()
                if not monthly_regional_external_features.empty
                else 0
            ),
            "earliest_export_year_month": (
                monthly_regional_external_features[["year", "month"]]
                .assign(month=lambda data: data["month"].astype(int).astype(str).str.zfill(2))
                .assign(year_month=lambda data: data["year"].astype(int).astype(str) + "-" + data["month"])
                ["year_month"]
                .min()
                if not monthly_regional_external_features.empty
                else pd.NA
            ),
            "latest_export_year_month": (
                monthly_regional_external_features[["year", "month"]]
                .assign(month=lambda data: data["month"].astype(int).astype(str).str.zfill(2))
                .assign(year_month=lambda data: data["year"].astype(int).astype(str) + "-" + data["month"])
                ["year_month"]
                .max()
                if not monthly_regional_external_features.empty
                else pd.NA
            )
        }
    ]
)

materialised_feature_sources = pd.DataFrame(
    [
        {
            "source_id": "cl_road_safety_annual_1972_2024",
            "default_handoff_scope": "national_annual_context",
            "optional_regional_scope": "national_annual_context_repeated_to_region",
            "monthly_rows_default": len(monthly_annual_context_features),
            "monthly_rows_regional_optional": len(monthly_regional_annual_context_features),
            "feature_columns": [
                column
                for column in monthly_annual_context_features.columns
                if column not in MONTHLY_JOIN_KEYS + ["country", "region", "source_id"]
            ]
        },
        {
            "source_id": "cl_weather_open_meteo_daily",
            "default_handoff_scope": "national_weather_context",
            "optional_regional_scope": "regional_weather_proxy",
            "monthly_rows_default": len(monthly_weather_features),
            "monthly_rows_regional_optional": len(monthly_regional_weather_features),
            "feature_columns": [
                column
                for column in monthly_weather_features.columns
                if column not in MONTHLY_JOIN_KEYS + ["country", "region", "source_id"]
            ]
        },
        {
            "source_id": "cl_public_holidays_nager_date",
            "default_handoff_scope": "national_calendar_context",
            "optional_regional_scope": "national_calendar_repeated_to_region",
            "monthly_rows_default": len(monthly_calendar_features),
            "monthly_rows_regional_optional": len(monthly_regional_calendar_features),
            "feature_columns": [
                column
                for column in monthly_calendar_features.columns
                if column not in MONTHLY_JOIN_KEYS + ["country", "region", "source_id"]
            ]
        }
    ]
)

monthly_external_feature_columns = pd.DataFrame(
    {
        "column_name": monthly_external_features.columns,
        "included_in_default_model_handoff": True
    }
)

monthly_regional_external_feature_columns = pd.DataFrame(
    {
        "column_name": monthly_regional_external_features.columns,
        "included_in_optional_regional_handoff": True
    }
)

display(monthly_feature_window)
display(regional_feature_coverage)
display(materialised_feature_sources)
display(monthly_external_feature_columns)
display(monthly_external_features.head())
display(monthly_regional_external_feature_columns)
display(monthly_regional_external_features.head())



,handoff_table,model_start_date,model_end_date,grain,join_keys,rows_before_model_window_filter,rows_after_model_window_filter,regions_in_export,earliest_export_year_month,latest_export_year_month
0,monthly_external_features,2021-01-01,2026-04-30,national_month,"[year, month]",648,64,0,2021-01,2026-04
1,monthly_regional_external_features,2021-01-01,2026-04-30,region_month_optional,"[region, year, month]",5832,576,9,2021-01,2026-04


,region,first_year,last_year,row_count
0,Antofagasta,2021,2026,64
1,Arica,2021,2026,64
2,Concepcion,2021,2026,64
3,La Serena,2021,2026,64
4,Puerto Montt,2021,2026,64
5,Punta Arenas,2021,2026,64
6,Santiago,2021,2026,64
7,Temuco,2021,2026,64
8,Valparaiso,2021,2026,64


,source_id,default_handoff_scope,optional_regional_scope,monthly_rows_default,monthly_rows_regional_optional,feature_columns
0,cl_road_safety_annual_1972_2024,national_annual_context,national_annual_context_repeated_to_region,636,5724,"[source_year, source_granularity, feature_lag_..."
1,cl_weather_open_meteo_daily,national_weather_context,regional_weather_proxy,64,448,"[avg_temperature, max_temperature, min_tempera..."
2,cl_public_holidays_nager_date,national_calendar_context,national_calendar_repeated_to_region,72,648,"[days_in_month, weekend_days, public_holidays,..."


,column_name,included_in_default_model_handoff
0,date,True
1,year,True
2,month,True
3,lag_1yr_national_annual_collisions_context,True
4,lag_1yr_national_annual_fatalities_context,True
5,lag_1yr_national_annual_serious_injuries_context,True
6,lag_1yr_national_annual_moderate_injuries_context,True
7,lag_1yr_national_annual_minor_injuries_context,True
8,lag_1yr_national_annual_total_injuries_context,True
9,lag_1yr_national_annual_total_victims_context,True


,date,year,month,lag_1yr_national_annual_collisions_context,lag_1yr_national_annual_fatalities_context,lag_1yr_national_annual_serious_injuries_context,lag_1yr_national_annual_moderate_injuries_context,lag_1yr_national_annual_minor_injuries_context,lag_1yr_national_annual_total_injuries_context,lag_1yr_national_annual_total_victims_context,...,avg_snowfall_sum,avg_precipitation_hours,max_wind_speed,max_wind_gust,avg_rain_days,weather_locations,days_in_month,weekend_days,public_holidays,working_days
576,2021-01-01,2021,1,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,43588.0,...,0.01,61.285714,56.5,110.9,7.285714,7.0,31.0,10.0,1.0,20.0
577,2021-02-01,2021,2,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,43588.0,...,0.00,29.142857,59.8,115.6,5.142857,7.0,28.0,8.0,0.0,20.0
578,2021-03-01,2021,3,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,43588.0,...,0.00,50.714286,66.3,126.7,6.857143,7.0,31.0,8.0,0.0,23.0
579,2021-04-01,2021,4,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,43588.0,...,0.00,71.857143,52.5,101.2,8.142857,7.0,30.0,8.0,2.0,21.0
580,2021-05-01,2021,5,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,43588.0,...,0.04,73.142857,46.4,90.7,8.714286,7.0,31.0,10.0,2.0,20.0


,column_name,included_in_optional_regional_handoff
0,date,True
1,year,True
2,month,True
3,region,True
4,lag_1yr_national_annual_collisions_context,True
5,lag_1yr_national_annual_fatalities_context,True
6,lag_1yr_national_annual_serious_injuries_context,True
7,lag_1yr_national_annual_moderate_injuries_context,True
8,lag_1yr_national_annual_minor_injuries_context,True
9,lag_1yr_national_annual_total_injuries_context,True


,date,year,month,region,lag_1yr_national_annual_collisions_context,lag_1yr_national_annual_fatalities_context,lag_1yr_national_annual_serious_injuries_context,lag_1yr_national_annual_moderate_injuries_context,lag_1yr_national_annual_minor_injuries_context,lag_1yr_national_annual_total_injuries_context,...,snowfall_sum,precipitation_hours,max_wind_speed,max_wind_gust,rain_days,weather_days,days_in_month,weekend_days,public_holidays,working_days
576,2021-01-01,2021,1,Antofagasta,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,...,0.0,2.0,23.2,49.3,1.0,31.0,31.0,10.0,1.0,20.0
577,2021-02-01,2021,2,Antofagasta,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,...,0.0,0.0,23.1,51.8,0.0,28.0,28.0,8.0,0.0,20.0
578,2021-03-01,2021,3,Antofagasta,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,...,0.0,0.0,21.6,46.8,0.0,31.0,31.0,8.0,0.0,23.0
579,2021-04-01,2021,4,Antofagasta,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,...,0.0,7.0,20.9,46.4,2.0,30.0,30.0,8.0,2.0,21.0
580,2021-05-01,2021,5,Antofagasta,64707.0,1485.0,6430.0,3378.0,32295.0,42103.0,...,0.0,5.0,19.5,42.8,3.0,31.0,31.0,10.0,2.0,20.0


# **External Data EDA and Presentation Plots**

These Plotly views use the fully preprocessed feature tables, so they reflect exactly what the forecasting handoff will contain. Each figure is saved as a standalone HTML artifact in the processed external-data folder for slides, demos or screenshots.


In [ ]:
PLOTLY_EDA_OUTPUT_DIRECTORY = Path(
    "/content/drive/MyDrive/EP/External Data/processed/eda_plotly"
)

PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLOR_SEQUENCE = [
    "#2563eb",
    "#16a34a",
    "#dc2626",
    "#9333ea",
    "#f59e0b",
    "#0891b2",
    "#db2777",
    "#4b5563"
]
PLOTLY_FIGURE_CONFIG = {
    "displayModeBar": True,
    "responsive": True
}

# Colab can show blank output areas with Plotly's default renderer.
# Inline HTML is the most reliable notebook display path; the full standalone
# HTML export below still embeds Plotly for sharing outside the notebook.
PLOTLY_INLINE_RENDER_MODE = "html"
pio.renderers.default = "colab"

PRESENTATION_SOURCE_LABELS = {
    "cl_road_safety_annual_1972_2024": "Road safety",
    "cl_weather_open_meteo_daily": "Weather",
    "cl_public_holidays_nager_date": "Calendar"
}

PRESENTATION_FEATURE_LABELS = {
    "collisions": "Collisions",
    "fatalities": "Fatalities",
    "serious_injuries": "Serious injuries",
    "total_injuries": "Total injuries",
    "total_victims": "Total victims",
    "vehicle_parc": "Vehicle parc",
    "population": "Population",
    "collisions_per_10k_vehicles": "Collisions per 10k vehicles",
    "fatalities_per_10k_vehicles": "Fatalities per 10k vehicles",
    "injuries_per_10k_vehicles": "Injuries per 10k vehicles",
    "collisions_per_100k_population": "Collisions per 100k population",
    "avg_temperature": "Average temperature",
    "max_temperature": "Max temperature",
    "min_temperature": "Min temperature",
    "avg_precipitation_sum": "Average precipitation",
    "avg_rain_sum": "Average rain",
    "avg_precipitation_hours": "Average precipitation hours",
    "max_wind_speed": "Max wind speed",
    "max_wind_gust": "Max wind gust",
    "avg_rain_days": "Average rain days",
    "rain_sum": "Rain",
    "precipitation_sum": "Precipitation",
    "working_days": "Working days",
    "weekend_days": "Weekend days",
    "public_holidays": "Public holidays",
    "days_in_month": "Days in month",
    "lag_1yr_national_annual_collisions_context": "Lagged collisions",
    "lag_1yr_national_annual_fatalities_context": "Lagged fatalities",
    "lag_1yr_national_annual_total_injuries_context": "Lagged total injuries",
    "lag_1yr_national_annual_vehicle_parc_context": "Lagged vehicle parc",
    "lag_1yr_national_annual_population_context": "Lagged population",
    "lag_1yr_national_annual_collisions_per_10k_vehicles_context": "Lagged collisions per 10k vehicles",
    "lag_1yr_national_annual_injuries_per_10k_vehicles_context": "Lagged injuries per 10k vehicles"
}

EDA_NON_FEATURE_COLUMNS = {
    "date",
    "year",
    "month",
    "country",
    "region",
    "source_id",
    "source_year",
    "source_granularity",
    "feature_lag_years",
    "feature_scope",
    "feature_semantics",
    "aggregation_rule",
    "weather_location_weight",
    "weather_days",
    "weather_locations"
}

MONTH_ORDER = list(range(1, 13))
MONTH_LABELS = {
    month: pd.Timestamp(year=2024, month=month, day=1).strftime("%b")
    for month in MONTH_ORDER
}

ANNUAL_CONTEXT_PRESENTATION_FEATURES = {
    "collisions": "Collisions",
    "total_injuries": "Total injuries",
    "fatalities": "Fatalities",
    "vehicle_parc": "Vehicle parc",
    "population": "Population",
    "collisions_per_10k_vehicles": "Collisions per 10k vehicles"
}

NATIONAL_WEATHER_PRESENTATION_FEATURES = {
    "avg_temperature": "Average temperature",
    "avg_rain_sum": "Average rain",
    "avg_rain_days": "Average rain days",
    "max_wind_gust": "Max wind gust"
}

CORRELATION_PREFERRED_FEATURES = [
    "lag_1yr_national_annual_collisions_context",
    "lag_1yr_national_annual_total_injuries_context",
    "lag_1yr_national_annual_vehicle_parc_context",
    "lag_1yr_national_annual_collisions_per_10k_vehicles_context",
    "avg_temperature",
    "avg_rain_sum",
    "avg_rain_days",
    "max_wind_gust",
    "working_days",
    "weekend_days",
    "public_holidays"
]


def prettify_feature_name(feature_name):
    label = PRESENTATION_FEATURE_LABELS.get(feature_name)

    if label:
        return label

    label = str(feature_name)
    label = label.replace("lag_1yr_national_annual_", "")
    label = label.replace("_context", "")
    label = label.replace("avg_", "average_")
    label = label.replace("_", " ")

    return label.title()


def ensure_month_date_column(
    dataframe,
    date_column="date"
):
    if dataframe.empty:
        return dataframe.copy()

    data = dataframe.copy()

    if date_column in data.columns:
        data[date_column] = pd.to_datetime(
            data[date_column],
            errors="coerce"
        )
        return data

    if {"year", "month"}.issubset(data.columns):
        year = pd.to_numeric(
            data["year"],
            errors="coerce"
        ).astype("Int64").astype(str)

        month = pd.to_numeric(
            data["month"],
            errors="coerce"
        ).astype("Int64").astype(str).str.zfill(2)

        data[date_column] = pd.to_datetime(
            year + "-" + month + "-01",
            errors="coerce"
        )

    return data


def valid_numeric_feature_columns(
    dataframe,
    excluded_columns=None,
    min_non_null=3
):
    excluded_columns = set(excluded_columns or [])
    valid_columns = []

    for column in dataframe.columns:
        if column in excluded_columns:
            continue

        numeric_series = pd.to_numeric(
            dataframe[column],
            errors="coerce"
        )

        if (
            numeric_series.notna().sum() >= min_non_null
            and numeric_series.nunique(dropna=True) > 1
        ):
            valid_columns.append(column)

    return valid_columns


def infer_feature_source_family(feature_name):
    if str(feature_name).startswith("lag_1yr_national_annual_"):
        return "Road safety"

    if feature_name in NATIONAL_WEATHER_PRESENTATION_FEATURES:
        return "Weather"

    if feature_name in {"working_days", "weekend_days", "public_holidays", "days_in_month"}:
        return "Calendar"

    return "Other"


def count_feature_list(value):
    if isinstance(value, list):
        return len(value)

    if pd.isna(value):
        return 0

    if isinstance(value, str):
        cleaned = (
            value
            .strip()
            .strip("[]")
        )
        if not cleaned:
            return 0
        return len(
            [
                item
                for item in cleaned.split(",")
                if item.strip()
            ]
        )

    return 0


def display_plotly_figure_inline(
    figure,
    render_mode=PLOTLY_INLINE_RENDER_MODE
):
    """
    Display Plotly figures reliably inside Google Colab.

    The normal fig.show() path can occasionally leave a blank output region in
    Colab. Rendering a Plotly HTML fragment through IPython display avoids that
    while preserving hover, zoom and toolbar interactivity.
    """

    if render_mode == "colab":
        figure.show(
            renderer="colab",
            config=PLOTLY_FIGURE_CONFIG
        )
        return

    display(
        HTML(
            figure.to_html(
                include_plotlyjs="cdn",
                full_html=False,
                config=PLOTLY_FIGURE_CONFIG
            )
        )
    )


def save_and_show_plotly_figure(
    figure,
    file_stem,
    output_directory=PLOTLY_EDA_OUTPUT_DIRECTORY
):
    output_directory = Path(output_directory)
    output_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    figure.update_layout(
        template=PLOTLY_TEMPLATE,
        colorway=PLOTLY_COLOR_SEQUENCE,
        font={
            "family": "Arial",
            "size": 14
        },
        title_x=0.02,
        title_xanchor="left",
        margin={
            "l": 70,
            "r": 40,
            "t": 90,
            "b": 70
        },
        legend_title_text=""
    )

    html_path = output_directory / f"{file_stem}.html"
    figure.write_html(
        str(html_path),
        include_plotlyjs=True,
        full_html=True,
        config=PLOTLY_FIGURE_CONFIG
    )
    display_plotly_figure_inline(
        figure
    )

    return {
        "plot_name": file_stem,
        "title": figure.layout.title.text,
        "html_path": str(html_path)
    }


def create_monthly_feature_eda_summary(
    monthly_features
):
    if monthly_features.empty:
        return pd.DataFrame()

    feature_columns = valid_numeric_feature_columns(
        monthly_features,
        excluded_columns=EDA_NON_FEATURE_COLUMNS,
        min_non_null=1
    )
    row_count = len(monthly_features)
    summary_rows = []

    for column in feature_columns:
        numeric_series = pd.to_numeric(
            monthly_features[column],
            errors="coerce"
        )
        observed = numeric_series.dropna()

        if observed.empty:
            continue

        summary_rows.append(
            {
                "source_family": infer_feature_source_family(column),
                "feature": column,
                "presentation_label": prettify_feature_name(column),
                "non_null_months": int(observed.shape[0]),
                "missing_months": int(row_count - observed.shape[0]),
                "missing_share": round(
                    1 - (observed.shape[0] / row_count),
                    3
                ),
                "min": observed.min(),
                "median": observed.median(),
                "max": observed.max(),
                "std": observed.std()
            }
        )

    summary = pd.DataFrame(summary_rows)

    if summary.empty:
        return summary

    return (
        summary
        .sort_values(["missing_share", "source_family", "presentation_label"])
        .reset_index(drop=True)
    )


def build_source_coverage_figure(
    materialised_feature_sources
):
    required_columns = {
        "source_id",
        "monthly_rows_default",
        "monthly_rows_regional_optional"
    }

    if (
        materialised_feature_sources.empty
        or not required_columns.issubset(materialised_feature_sources.columns)
    ):
        return None

    coverage = materialised_feature_sources.copy()
    coverage["source_label"] = (
        coverage["source_id"]
        .map(PRESENTATION_SOURCE_LABELS)
        .fillna(coverage["source_id"])
    )
    coverage["feature_count"] = coverage.get(
        "feature_columns",
        pd.Series(index=coverage.index, dtype=object)
    ).apply(count_feature_list)

    for column in ["monthly_rows_default", "monthly_rows_regional_optional"]:
        coverage[column] = pd.to_numeric(
            coverage[column],
            errors="coerce"
        ).fillna(0)

    plot_data = coverage.melt(
        id_vars=["source_id", "source_label", "feature_count"],
        value_vars=["monthly_rows_default", "monthly_rows_regional_optional"],
        var_name="handoff_scope",
        value_name="monthly_rows"
    )
    plot_data["handoff_scope"] = plot_data["handoff_scope"].map(
        {
            "monthly_rows_default": "National handoff",
            "monthly_rows_regional_optional": "Regional optional"
        }
    )

    figure = px.bar(
        plot_data,
        x="source_label",
        y="monthly_rows",
        color="handoff_scope",
        barmode="group",
        text="monthly_rows",
        hover_data={
            "source_id": True,
            "feature_count": True,
            "monthly_rows": ":,.0f"
        },
        labels={
            "source_label": "External source",
            "monthly_rows": "Monthly rows",
            "handoff_scope": "Handoff scope"
        },
        title="External Feature Coverage by Source"
    )
    figure.update_traces(
        texttemplate="%{text:.0f}",
        textposition="outside"
    )
    figure.update_layout(
        height=540,
        yaxis_title="Rows available after preprocessing"
    )

    return figure


def build_annual_context_figure(
    annual_feature_table
):
    if annual_feature_table.empty or "year" not in annual_feature_table.columns:
        return None

    annual = annual_feature_table.copy()
    annual["year"] = pd.to_numeric(
        annual["year"],
        errors="coerce"
    )
    annual = annual.dropna(
        subset=["year"]
    )

    if annual.empty:
        return None

    annual["year"] = annual["year"].astype(int)
    indexed_frames = []

    for column, label in ANNUAL_CONTEXT_PRESENTATION_FEATURES.items():
        if column not in annual.columns:
            continue

        feature_by_year = annual[["year", column]].copy()
        feature_by_year[column] = pd.to_numeric(
            feature_by_year[column],
            errors="coerce"
        )
        feature_by_year = (
            feature_by_year
            .dropna(subset=[column])
            .groupby("year", as_index=False)[column]
            .mean()
            .sort_values("year")
        )

        non_zero = feature_by_year.loc[
            feature_by_year[column] != 0
        ]

        if non_zero.empty:
            continue

        baseline_year = int(non_zero.iloc[0]["year"])
        baseline_value = non_zero.iloc[0][column]
        feature_by_year["indexed_value"] = (
            feature_by_year[column]
            / baseline_value
            * 100
        )
        feature_by_year["raw_value"] = feature_by_year[column]
        feature_by_year["baseline_year"] = baseline_year
        feature_by_year["feature_label"] = label

        indexed_frames.append(
            feature_by_year[
                [
                    "year",
                    "feature_label",
                    "indexed_value",
                    "raw_value",
                    "baseline_year"
                ]
            ]
        )

    if not indexed_frames:
        return None

    plot_data = pd.concat(
        indexed_frames,
        ignore_index=True
    )

    figure = px.line(
        plot_data,
        x="year",
        y="indexed_value",
        color="feature_label",
        markers=True,
        hover_data={
            "feature_label": False,
            "indexed_value": ":.1f",
            "raw_value": ":,.2f",
            "baseline_year": True
        },
        labels={
            "year": "Year",
            "indexed_value": "Index"
        },
        title="Road-Safety Context Trend Indexed to First Available Year"
    )
    figure.add_hline(
        y=100,
        line_dash="dot",
        line_color="#64748b"
    )
    figure.update_traces(
        line_width=3,
        marker_size=7
    )
    figure.update_layout(
        height=560,
        yaxis_title="Index, first available year = 100"
    )
    figure.update_xaxes(
        dtick=5
    )

    return figure


def build_weather_feature_figure(
    monthly_weather_features
):
    if monthly_weather_features.empty:
        return None

    weather = ensure_month_date_column(
        monthly_weather_features
    )
    feature_pairs = [
        (column, label)
        for column, label in NATIONAL_WEATHER_PRESENTATION_FEATURES.items()
        if column in weather.columns
    ]

    if not feature_pairs:
        return None

    column_count = 2 if len(feature_pairs) > 1 else 1
    row_count = (
        len(feature_pairs)
        + column_count
        - 1
    ) // column_count

    figure = make_subplots(
        rows=row_count,
        cols=column_count,
        subplot_titles=[
            label
            for _, label in feature_pairs
        ],
        horizontal_spacing=0.10,
        vertical_spacing=0.18
    )

    for index, (column, label) in enumerate(feature_pairs):
        row_number = (index // column_count) + 1
        column_number = (index % column_count) + 1
        plot_data = weather[["date", column]].copy()
        plot_data[column] = pd.to_numeric(
            plot_data[column],
            errors="coerce"
        )
        plot_data = plot_data.dropna(
            subset=["date", column]
        )

        figure.add_trace(
            go.Scatter(
                x=plot_data["date"],
                y=plot_data[column],
                mode="lines+markers",
                name=label,
                line={
                    "color": PLOTLY_COLOR_SEQUENCE[index % len(PLOTLY_COLOR_SEQUENCE)],
                    "width": 3
                },
                marker={
                    "size": 6
                },
                hovertemplate=(
                    "%{x|%b %Y}<br>"
                    f"{label}: %{{y:,.2f}}"
                    "<extra></extra>"
                )
            ),
            row=row_number,
            col=column_number
        )
        figure.update_yaxes(
            title_text=label,
            row=row_number,
            col=column_number
        )

    figure.update_layout(
        title="National Monthly Weather Signals",
        height=max(560, 300 * row_count),
        showlegend=False
    )
    figure.update_xaxes(
        tickformat="%b %Y"
    )

    return figure


def build_regional_weather_heatmap(
    monthly_regional_weather_features
):
    if (
        monthly_regional_weather_features.empty
        or "region" not in monthly_regional_weather_features.columns
        or "month" not in monthly_regional_weather_features.columns
    ):
        return None

    value_column = next(
        (
            column
            for column in [
                "rain_sum",
                "precipitation_sum",
                "max_wind_gust",
                "max_wind_speed"
            ]
            if column in monthly_regional_weather_features.columns
        ),
        None
    )

    if value_column is None:
        return None

    regional = monthly_regional_weather_features.copy()
    regional["month"] = pd.to_numeric(
        regional["month"],
        errors="coerce"
    )
    regional[value_column] = pd.to_numeric(
        regional[value_column],
        errors="coerce"
    )
    regional = regional.dropna(
        subset=["region", "month", value_column]
    )

    if regional.empty:
        return None

    regional["month"] = regional["month"].astype(int)
    regional_pivot = regional.pivot_table(
        index="region",
        columns="month",
        values=value_column,
        aggfunc="mean"
    )
    regional_pivot = regional_pivot.reindex(
        columns=MONTH_ORDER
    )
    value_label = prettify_feature_name(value_column)

    figure = go.Figure(
        data=go.Heatmap(
            z=regional_pivot.values,
            x=[
                MONTH_LABELS[month]
                for month in regional_pivot.columns
            ],
            y=regional_pivot.index,
            colorscale="Blues",
            colorbar={
                "title": value_label
            },
            hovertemplate=(
                "Region: %{y}<br>"
                "Month: %{x}<br>"
                f"{value_label}: %{{z:,.1f}}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title=f"Regional Weather Seasonality - {value_label}",
        height=620,
        xaxis_title="Month",
        yaxis_title="Region"
    )

    return figure


def build_calendar_working_days_figure(
    monthly_calendar_features
):
    if monthly_calendar_features.empty:
        return None

    calendar = ensure_month_date_column(
        monthly_calendar_features
    )
    required_columns = {
        "date",
        "working_days",
        "weekend_days"
    }

    if not required_columns.issubset(calendar.columns):
        return None

    for column in ["working_days", "weekend_days", "public_holidays"]:
        if column in calendar.columns:
            calendar[column] = pd.to_numeric(
                calendar[column],
                errors="coerce"
            )

    figure = make_subplots(
        specs=[
            [
                {
                    "secondary_y": True
                }
            ]
        ]
    )
    figure.add_trace(
        go.Bar(
            x=calendar["date"],
            y=calendar["working_days"],
            name="Working days",
            marker_color="#16a34a",
            hovertemplate="Working days: %{y}<extra></extra>"
        ),
        secondary_y=False
    )
    figure.add_trace(
        go.Bar(
            x=calendar["date"],
            y=calendar["weekend_days"],
            name="Weekend days",
            marker_color="#94a3b8",
            hovertemplate="Weekend days: %{y}<extra></extra>"
        ),
        secondary_y=False
    )

    if "public_holidays" in calendar.columns:
        figure.add_trace(
            go.Scatter(
                x=calendar["date"],
                y=calendar["public_holidays"],
                name="Public holidays",
                mode="lines+markers",
                line={
                    "color": "#dc2626",
                    "width": 3
                },
                marker={
                    "size": 7
                },
                hovertemplate="Public holidays: %{y}<extra></extra>"
            ),
            secondary_y=True
        )

    figure.update_layout(
        title="Calendar Structure for Monthly Forecasting",
        height=560,
        barmode="stack",
        xaxis_title="Month"
    )
    figure.update_yaxes(
        title_text="Working and weekend days",
        secondary_y=False
    )
    figure.update_yaxes(
        title_text="Public holidays",
        secondary_y=True,
        rangemode="tozero"
    )
    figure.update_xaxes(
        tickformat="%b %Y"
    )

    return figure


def build_feature_correlation_figure(
    monthly_external_features,
    max_features=16
):
    if monthly_external_features.empty:
        return None

    valid_columns = valid_numeric_feature_columns(
        monthly_external_features,
        excluded_columns=EDA_NON_FEATURE_COLUMNS,
        min_non_null=6
    )

    selected_columns = [
        column
        for column in CORRELATION_PREFERRED_FEATURES
        if column in valid_columns
    ]
    selected_columns.extend(
        [
            column
            for column in valid_columns
            if column not in selected_columns
        ][: max_features - len(selected_columns)]
    )
    selected_columns = selected_columns[:max_features]

    if len(selected_columns) < 2:
        return None

    correlation_data = monthly_external_features[selected_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )
    correlation_matrix = correlation_data.corr()

    if correlation_matrix.empty:
        return None

    labels = [
        prettify_feature_name(column)
        for column in correlation_matrix.columns
    ]

    figure = go.Figure(
        data=go.Heatmap(
            z=correlation_matrix.values,
            x=labels,
            y=labels,
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            reversescale=True,
            colorbar={
                "title": "Correlation"
            },
            hovertemplate=(
                "%{y}<br>"
                "%{x}<br>"
                "Correlation: %{z:.2f}"
                "<extra></extra>"
            )
        )
    )
    figure.update_layout(
        title="Monthly External Feature Correlations",
        height=760,
        xaxis_tickangle=-35
    )

    return figure


def build_feature_completeness_figure(
    feature_summary,
    max_features=24
):
    if feature_summary.empty:
        return None

    plot_data = feature_summary.copy()
    plot_data["completion_share"] = (
        1 - plot_data["missing_share"]
    )
    plot_data = (
        plot_data
        .sort_values(["completion_share", "presentation_label"])
        .tail(max_features)
    )

    figure = px.bar(
        plot_data,
        x="completion_share",
        y="presentation_label",
        color="source_family",
        orientation="h",
        hover_data={
            "feature": True,
            "non_null_months": True,
            "missing_months": True,
            "completion_share": ":.0%"
        },
        labels={
            "completion_share": "Completion",
            "presentation_label": "Feature",
            "source_family": "Source"
        },
        title="External Feature Completeness in the Model Window"
    )
    figure.update_layout(
        height=720,
        xaxis_tickformat=".0%"
    )

    return figure


In [ ]:
eda_monthly_feature_summary = create_monthly_feature_eda_summary(
    monthly_external_features
)

display(
    eda_monthly_feature_summary
)

eda_plotly_figures = [
    (
        "external_feature_coverage_by_source",
        build_source_coverage_figure(
            materialised_feature_sources
        )
    ),
    (
        "road_safety_context_indexed_trends",
        build_annual_context_figure(
            annual_feature_table
        )
    ),
    (
        "national_monthly_weather_signals",
        build_weather_feature_figure(
            monthly_weather_features
        )
    ),
    (
        "regional_weather_seasonality_heatmap",
        build_regional_weather_heatmap(
            monthly_regional_weather_features
        )
    ),
    (
        "calendar_working_days_and_holidays",
        build_calendar_working_days_figure(
            monthly_calendar_features
        )
    ),
    (
        "monthly_external_feature_correlations",
        build_feature_correlation_figure(
            monthly_external_features
        )
    ),
    (
        "external_feature_completeness",
        build_feature_completeness_figure(
            eda_monthly_feature_summary
        )
    )
]

eda_plotly_artifacts = []

for file_stem, figure in eda_plotly_figures:
    if figure is None:
        print(f"Skipped {file_stem}: not enough processed data for this view.")
        continue

    eda_plotly_artifacts.append(
        save_and_show_plotly_figure(
            figure=figure,
            file_stem=file_stem,
            output_directory=PLOTLY_EDA_OUTPUT_DIRECTORY
        )
    )

eda_plotly_artifacts = pd.DataFrame(
    eda_plotly_artifacts
)

display(
    eda_plotly_artifacts
)


In [ ]:
def create_feature_inventory_documents(
    feature_inventory
):
    """
    Create searchable documents describing model-ready features.
    """

    documents = []

    if feature_inventory.empty:
        return documents

    for _, row in feature_inventory.iterrows():

        page_content = (
            f"Feature: {row.get('standard_feature_name')}\n"
            f"Original source column: {row.get('original_column_name')}\n"
            f"Source id: {row.get('source_id')}\n"
            f"Source name: {row.get('source_name')}\n"
            f"Table: {row.get('table_name')}\n"
            f"Data type: {row.get('pandas_dtype')}\n"
            f"Numeric values: {row.get('numeric_count')}\n"
            f"Signal category: {row.get('forecast_signal_category')}\n"
            f"Join keys: {row.get('join_keys')}\n"
            f"Target model use: {row.get('target_model_use')}"
        )

        metadata = {
            "source_id": row.get("source_id"),
            "source_type": "external_forecasting_signal",
            "record_type": "feature_inventory",
            "feature_name": row.get("standard_feature_name"),
            "original_column_name": row.get("original_column_name"),
            "forecast_signal_category": row.get("forecast_signal_category"),
            "target_model_use": row.get("target_model_use")
        }

        documents.append(
            Document(
                page_content=page_content,
                metadata=metadata
            )
        )

    return documents


def create_monthly_feature_table_document(
    monthly_external_features
):
    """
    Create one searchable document summarising the model handoff table.
    """

    if monthly_external_features.empty:
        return []

    feature_columns = [
        column
        for column in monthly_external_features.columns
        if column not in ["country", "region", "year", "month"]
    ]

    page_content = (
        "Model handoff table: monthly_external_features\n"
        f"Rows: {len(monthly_external_features)}\n"
        f"Join keys: country, region, year, month\n"
        f"Feature columns: {', '.join(feature_columns)}\n"
        "Usage: merge this table into monthly spare-parts demand data before model training or scoring."
    )

    return [
        Document(
            page_content=page_content,
            metadata={
                "source_type": "model_handoff",
                "record_type": "monthly_feature_table",
                "table_name": "monthly_external_features",
                "join_keys": "country, region, year, month",
                "row_count": int(len(monthly_external_features))
            }
        )
    ]


feature_inventory_documents = create_feature_inventory_documents(
    feature_inventory_all_sources
)

monthly_feature_documents = create_monthly_feature_table_document(
    monthly_external_features
)

all_vector_documents = (
    registry_documents
    + all_summary_documents
    + all_row_documents
    + feature_inventory_documents
    + monthly_feature_documents
)

print("Feature inventory documents:", len(feature_inventory_documents))
print("Monthly feature documents:", len(monthly_feature_documents))
print("All vector documents after feature docs:", len(all_vector_documents))

Feature inventory documents: 20
Monthly feature documents: 1
All vector documents after feature docs: 78


# **Quality Checks**

These checks validate the tables we are about to pass to retrieval or modelling. The goal is to catch leakage, duplicated keys, accidental 12x annual aggregation, missing standard names and stale tests that only inspect the original crash summary.

In [ ]:
def assert_required_columns(
    dataframe,
    required_columns,
    table_name
):
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    assert not missing_columns, (
        f"{table_name} is missing required columns: {missing_columns}"
    )


def assert_unique_keys(
    dataframe,
    key_columns,
    table_name
):
    duplicate_count = dataframe.duplicated(
        subset=key_columns
    ).sum()

    assert duplicate_count == 0, (
        f"{table_name} has {duplicate_count} duplicate key rows for {key_columns}"
    )


def assert_annual_context_is_lagged(
    monthly_context,
    lag_years=1
):
    assert_required_columns(
        monthly_context,
        ["year", "source_year", "month", "feature_semantics", "aggregation_rule"],
        "monthly_annual_context_features"
    )

    lag_difference = (
        monthly_context["year"]
        - monthly_context["source_year"]
    )

    assert (lag_difference == lag_years).all(), (
        "Annual context leakage risk: forecast year must equal source_year + lag_years."
    )

    assert (
        monthly_context["feature_semantics"]
        == "repeated_annual_context"
    ).all(), "Annual context rows must be labelled as repeated_annual_context."

    assert (
        monthly_context["aggregation_rule"]
        == "first_or_mean_not_sum"
    ).all(), "Annual context rows must warn consumers not to sum repeated annual values."


def assert_annual_context_has_12_months_per_source_year(
    monthly_context
):
    group_columns = [
        column
        for column in ["country", "region", "source_id", "source_year"]
        if column in monthly_context.columns
    ]

    month_counts = (
        monthly_context
        .groupby(group_columns)
        ["month"]
        .nunique()
    )

    assert (month_counts == 12).all(), (
        "Each annual source row should create exactly 12 monthly context rows."
    )


def assert_no_unmapped_source_columns(
    feature_inventory
):
    unmapped = feature_inventory.loc[
        feature_inventory["original_column_name"]
        == feature_inventory["standard_feature_name"],
        "original_column_name"
    ].tolist()

    allowed_unmapped = {
        "source_id",
        "country",
        "region",
        "granularity",
        "table_name"
    }

    unexpected_unmapped = [
        column
        for column in unmapped
        if column not in allowed_unmapped
    ]

    assert not unexpected_unmapped, (
        f"These source columns are not mapped to model-ready names: {unexpected_unmapped}"
    )


def assert_monthly_handoff_table(
    monthly_external_features
):
    assert_required_columns(
        monthly_external_features,
        ["year", "month"],
        "monthly_external_features"
    )

    assert "region" not in monthly_external_features.columns, (
        "Default monthly_external_features should be national monthly grain, not regional."
    )

    assert (
        monthly_external_features["month"].between(1, 12).all()
    ), "monthly_external_features contains invalid month values."

    assert_unique_keys(
        monthly_external_features,
        MONTHLY_JOIN_KEYS,
        "monthly_external_features"
    )


def assert_optional_regional_handoff_table(
    monthly_regional_external_features
):
    if monthly_regional_external_features.empty:
        return

    assert_required_columns(
        monthly_regional_external_features,
        ["region", "year", "month"],
        "monthly_regional_external_features"
    )

    assert_unique_keys(
        monthly_regional_external_features,
        REGIONAL_MONTHLY_JOIN_KEYS,
        "monthly_regional_external_features"
    )


def assert_vector_documents_cover_registered_sources(
    all_vector_documents,
    source_registry
):
    registered_source_ids = set(
        source_registry["source_id"]
    )
    vector_source_ids = {
        document.metadata.get("source_id")
        for document in all_vector_documents
        if document.metadata.get("source_id")
    }

    missing_source_ids = (
        registered_source_ids
        - vector_source_ids
    )

    assert not missing_source_ids, (
        f"Vector documents missing registered source ids: {missing_source_ids}"
    )


def run_pipeline_quality_checks():
    assert_required_columns(
        feature_inventory_all_sources,
        [
            "source_id",
            "original_column_name",
            "standard_feature_name",
            "numeric_count",
            "forecast_signal_category"
        ],
        "feature_inventory_all_sources"
    )

    assert_no_unmapped_source_columns(
        feature_inventory_all_sources
    )

    assert_required_columns(
        annual_feature_table,
        ["year", "country", "region", "source_id"],
        "annual_feature_table"
    )

    assert_required_columns(
        monthly_annual_context_features,
        ["country", "region", "year", "month", "source_year", "feature_scope"],
        "monthly_annual_context_features"
    )

    assert_annual_context_is_lagged(
        monthly_annual_context_features,
        lag_years=1
    )

    assert_annual_context_has_12_months_per_source_year(
        monthly_annual_context_features
    )

    assert_monthly_handoff_table(
        monthly_external_features
    )

    assert_optional_regional_handoff_table(
        monthly_regional_external_features
    )

    assert_vector_documents_cover_registered_sources(
        all_vector_documents,
        source_registry
    )

    print("Pipeline quality checks passed.")


run_pipeline_quality_checks()


Pipeline quality checks passed.


# **Export Model Handoff Files**

These are the files we create from the external sources for the forecasting models. The CSV is easiest for review; Parquet is better for modelling pipelines when the runtime supports it.

The export files are for the team to inspect, share or merge into model data. The embed/store section later is for the LLM layer, not for model training directly.

In [ ]:
OUTPUT_DIRECTORY = Path(
    "/content/drive/MyDrive/EP/External Data/processed"
)


def export_model_handoff_files(
    monthly_external_features,
    monthly_regional_external_features,
    feature_inventory,
    source_registry,
    materialised_feature_sources,
    output_directory
):
    """
    Export model handoff tables for team and downstream pipelines.
    """

    output_directory = Path(output_directory)
    output_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    outputs = {}

    monthly_csv_path = output_directory / "monthly_external_features.csv"
    monthly_regional_csv_path = output_directory / "monthly_regional_external_features.csv"
    inventory_csv_path = output_directory / "feature_inventory_all_sources.csv"
    registry_csv_path = output_directory / "external_source_registry.csv"
    materialised_sources_csv_path = output_directory / "materialised_feature_sources.csv"

    monthly_external_features.to_csv(
        monthly_csv_path,
        index=False
    )
    monthly_regional_external_features.to_csv(
        monthly_regional_csv_path,
        index=False
    )
    feature_inventory.to_csv(
        inventory_csv_path,
        index=False
    )
    source_registry.to_csv(
        registry_csv_path,
        index=False
    )
    materialised_feature_sources.to_csv(
        materialised_sources_csv_path,
        index=False
    )

    outputs["monthly_external_features_csv"] = str(monthly_csv_path)
    outputs["monthly_regional_external_features_csv"] = str(monthly_regional_csv_path)
    outputs["feature_inventory_csv"] = str(inventory_csv_path)
    outputs["source_registry_csv"] = str(registry_csv_path)
    outputs["materialised_feature_sources_csv"] = str(materialised_sources_csv_path)

    try:
        monthly_parquet_path = output_directory / "monthly_external_features.parquet"
        monthly_external_features.to_parquet(
            monthly_parquet_path,
            index=False
        )
        outputs["monthly_external_features_parquet"] = str(monthly_parquet_path)
    except Exception as error:
        outputs["monthly_external_features_parquet_error"] = str(error)

    try:
        monthly_regional_parquet_path = output_directory / "monthly_regional_external_features.parquet"
        monthly_regional_external_features.to_parquet(
            monthly_regional_parquet_path,
            index=False
        )
        outputs["monthly_regional_external_features_parquet"] = str(monthly_regional_parquet_path)
    except Exception as error:
        outputs["monthly_regional_external_features_parquet_error"] = str(error)

    return outputs


exported_files = export_model_handoff_files(
    monthly_external_features=monthly_external_features,
    monthly_regional_external_features=monthly_regional_external_features,
    feature_inventory=feature_inventory_all_sources,
    source_registry=source_registry,
    materialised_feature_sources=materialised_feature_sources,
    output_directory=OUTPUT_DIRECTORY
)

display(pd.DataFrame(
    exported_files.items(),
    columns=["artifact", "path_or_status"]
))


,artifact,path_or_status
0,monthly_external_features_csv,/content/drive/MyDrive/EP/External Data/proces...
1,monthly_regional_external_features_csv,/content/drive/MyDrive/EP/External Data/proces...
2,feature_inventory_csv,/content/drive/MyDrive/EP/External Data/proces...
3,source_registry_csv,/content/drive/MyDrive/EP/External Data/proces...
4,materialised_feature_sources_csv,/content/drive/MyDrive/EP/External Data/proces...
5,monthly_external_features_parquet,/content/drive/MyDrive/EP/External Data/proces...
6,monthly_regional_external_features_parquet,/content/drive/MyDrive/EP/External Data/proces...


# **Embed and Store**

In [ ]:
#Using an embedding model that can handle multiple languages
embedding_model = HuggingFaceEmbeddings(
    model_name=(
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    )
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Create a fresh in-memory vector store

collection_name = "external_forecasting_signals"

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embedding_model
)

vector_store.reset_collection()

vector_store.add_documents(
    documents=all_vector_documents
)

print("Prepared:", len(all_vector_documents))
print("Stored:", vector_store._collection.count())

Prepared: 78
Stored: 78


# **Embedding Sanity Checks**

In [ ]:
#Check the model works
test_vector = embedding_model.embed_query(
    "road collisions in recent years"
)

print(f"Vector length: {len(test_vector)}")
print(test_vector[:5])

Vector length: 384
[0.17404012382030487, -0.39290332794189453, 0.25858378410339355, 0.198801189661026, -0.1088772639632225]


In [ ]:
print(
    "All vector documents prepared:",
    len(all_vector_documents)
)

print(
    "Registry documents:",
    len(registry_documents)
)

print(
    "Dataset summaries:",
    len(all_summary_documents)
)

print(
    "Row-level records:",
    len(all_row_documents)
)

print(
    "Feature inventory documents:",
    len(feature_inventory_documents)
)

print(
    "Monthly feature documents:",
    len(monthly_feature_documents)
)

print(
    "Vector documents stored:",
    vector_store._collection.count()
)

All vector documents prepared: 78
Registry documents: 3
Dataset summaries: 1
Row-level records: 53
Feature inventory documents: 20
Monthly feature documents: 1
Vector documents stored: 78


# **Retrieval Smoke Tests**

These tests check retrieval by intent, not just whether the old crash summary comes back. They should cover source discovery, row-level period lookup and forecasting-usefulness questions.

In [ ]:
retrieval_test_cases = [
    {
        "query": "Which sources could help forecast monthly collision spare-parts demand?",
        "expected_metadata": {
            "source_type": "external_forecasting_signal"
        }
    },
    {
        "query": "Find row-level Chile traffic data for 2024",
        "expected_metadata": {
            "record_type": "table_row",
            "year": 2024
        },
        "search_filter": {
            "record_type": "table_row",
            "year": 2024
        }
    },
    {
        "query": "Which source has annual vehicle parc and population context?",
        "expected_metadata": {
            "forecast_signal_category": "collision_frequency"
        },
        "search_filter": {
            "forecast_signal_category": "collision_frequency"
        }
    },
    {
        "query": "Which API source can create monthly weather features for Chile?",
        "expected_metadata": {
            "record_type": "source_registry",
            "source_id": "cl_weather_open_meteo_daily"
        },
        "search_filter": {
            "record_type": "source_registry",
            "source_id": "cl_weather_open_meteo_daily"
        }
    },
    {
        "query": "What table should the team merge into monthly forecasting models?",
        "expected_metadata": {
            "record_type": "monthly_feature_table",
            "table_name": "monthly_external_features"
        },
        "search_filter": {
            "record_type": "monthly_feature_table",
            "table_name": "monthly_external_features"
        }
    }
]


def metadata_matches(
    metadata,
    expected_metadata
):
    for key, expected_value in expected_metadata.items():
        if metadata.get(key) != expected_value:
            return False

    return True


def build_chroma_where_filter(
    metadata_filter
):
    """
    Convert a simple metadata filter into Chroma's where syntax.

    Chroma expects a filter with exactly one top-level operator when multiple
    conditions are used, so {"record_type": "x", "year": 2024} becomes:
    {"$and": [{"record_type": {"$eq": "x"}}, {"year": {"$eq": 2024}}]}
    """

    if not metadata_filter:
        return None

    conditions = [
        {
            key: {
                "$eq": value
            }
        }
        for key, value in metadata_filter.items()
    ]

    if len(conditions) == 1:
        return conditions[0]

    return {
        "$and": conditions
    }


def run_retrieval_smoke_tests(
    vector_store,
    retrieval_test_cases,
    k=8
):
    """
    Test both broad semantic retrieval and exact metadata-addressable records.
    """

    for test_case in retrieval_test_cases:

        search_filter = build_chroma_where_filter(
            test_case.get("search_filter")
        )

        if search_filter:
            results = vector_store.similarity_search(
                query=test_case["query"],
                k=k,
                filter=search_filter
            )
        else:
            results = vector_store.similarity_search(
                query=test_case["query"],
                k=k
            )

        assert results, (
            f"No retrieval results for query: {test_case['query']}"
        )

        matched = any(
            metadata_matches(
                result.metadata,
                test_case["expected_metadata"]
            )
            for result in results
        )

        assert matched, (
            "Retrieval test failed.\n"
            f"Query: {test_case['query']}\n"
            f"Search filter: {search_filter}\n"
            f"Expected metadata: {test_case['expected_metadata']}\n"
            f"Returned metadata: {[result.metadata for result in results]}"
        )

    print("Retrieval smoke tests passed.")


run_retrieval_smoke_tests(
    vector_store=vector_store,
    retrieval_test_cases=retrieval_test_cases,
    k=8
)

Retrieval smoke tests passed.


# **LLM Pipeline Q&A**

This optional section lets the team ask questions about the external-data pipeline in plain English.

The retrieval-only cell below does not need an API key and is safe to run in a demo. The LLM cell after that is commented and ready for someone to enable once they have added an OpenAI API key.

The LLM should not be treated as the forecasting model. Its job is to explain the sources, feature tables, join keys, usefulness ratings, limitations and leakage risks using the documents stored in Chroma.

In [ ]:
def retrieve_pipeline_context(
    question,
    vector_store,
    k=8,
    search_filter=None
):
    """
    Retrieve relevant pipeline documents for an LLM or for manual review.
    """

    if search_filter:
        return vector_store.similarity_search(
            query=question,
            k=k,
            filter=search_filter
        )

    return vector_store.similarity_search(
        query=question,
        k=k
    )


def format_retrieved_context(
    documents
):
    """
    Format retrieved documents so the team can inspect what the LLM would see.
    """

    context_blocks = []

    for number, document in enumerate(documents, start=1):
        context_blocks.append(
            (
                f"DOCUMENT {number}\n"
                f"Metadata: {document.metadata}\n"
                f"{document.page_content}"
            )
        )

    return "\n\n---\n\n".join(context_blocks)


def answer_pipeline_question_without_llm(
    question,
    vector_store,
    k=8,
    search_filter=None
):
    """
    Retrieval-only fallback. Useful if no LLM/API key is configured.
    """

    retrieved_documents = retrieve_pipeline_context(
        question=question,
        vector_store=vector_store,
        k=k,
        search_filter=search_filter
    )

    print("QUESTION")
    print(question)
    print("\nRETRIEVED CONTEXT")
    print(format_retrieved_context(retrieved_documents))

    return retrieved_documents


example_question = (
    "Which external sources are useful for monthly forecasting, "
    "and which ones are only annual context?"
)

_ = answer_pipeline_question_without_llm(
    question=example_question,
    vector_store=vector_store,
    k=5
)

QUESTION
Which external sources are useful for monthly forecasting, and which ones are only annual context?

RETRIEVED CONTEXT
DOCUMENT 1
Metadata: {'join_keys': 'country, region, year, month', 'file_type': 'api', 'source_name': 'Open-Meteo daily historical weather for Chile locations', 'country': 'Chile', 'region': 'location', 'source_type': 'external_forecasting_signal', 'forecasting_usefulness': 'high', 'record_type': 'source_registry', 'domain': 'weather', 'source_id': 'cl_weather_open_meteo_daily', 'forecast_signal_category': 'weather_collision_risk', 'status': 'candidate_api', 'usefulness_reason': 'The source can be materialised at daily level and aggregated to month, which matches the forecasting grain. Weather also has a plausible relationship with collision frequency.', 'target_model_use': 'monthly external regressor', 'granularity': 'daily_to_monthly'}
Source id: cl_weather_open_meteo_daily
Source name: Open-Meteo daily historical weather for Chile locations
Domain: weather
C

In [ ]:
# Optional LLM answer layer.
#
# Retrieval above works without an API key. To enable the LLM answer:
#
# 1. Add your key as a Colab secret called OPENAI_API_KEY, or uncomment the
#    placeholder below and paste a temporary key for local testing only.
# 2. Keep the example call commented until you actually want to spend API usage.
#
# Never commit or share a notebook with a real API key written into a cell.

import os

# API key placeholder for local testing only.
# os.environ["OPENAI_API_KEY"] = "paste-your-api-key-here"

LLM_CALL_LIMIT = 5
llm_calls_used = 0


def check_llm_call_limit():
    """
    Simple notebook-level safety guard so a demo cannot accidentally make
    unlimited LLM calls.
    """

    global llm_calls_used

    if llm_calls_used >= LLM_CALL_LIMIT:
        raise RuntimeError(
            f"LLM call limit reached for this notebook run: {LLM_CALL_LIMIT}"
        )

    llm_calls_used += 1


def answer_pipeline_question_with_llm(
    question,
    vector_store,
    model_name="gpt-4o-mini",
    k=8,
    search_filter=None
):
    """
    Ask an LLM to answer using retrieved pipeline context.

    Keep the answer grounded in the retrieved documents. The LLM explains the
    pipeline; it does not replace the forecasting model.
    """

    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Run the retrieval-only cell above, "
            "or add a key before using the LLM answer function."
        )

    check_llm_call_limit()

    from langchain_openai import ChatOpenAI

    retrieved_documents = retrieve_pipeline_context(
        question=question,
        vector_store=vector_store,
        k=k,
        search_filter=search_filter
    )

    context = format_retrieved_context(
        retrieved_documents
    )

    prompt = (
        "You are helping explain an external-data pipeline for monthly "
        "collision spare-parts forecasting.\n\n"
        "Use only the retrieved context. If the context does not contain the "
        "answer, say what is missing. Be clear about whether something is for "
        "the forecasting model or for the LLM/search layer.\n\n"
        f"Question:\n{question}\n\n"
        f"Retrieved context:\n{context}"
    )

    llm = ChatOpenAI(
        model=model_name,
        temperature=0
    )

    response = llm.invoke(prompt)
    print(response.content)

    return response, retrieved_documents


# Example LLM call. Leave commented unless an API key is configured.
# response, docs = answer_pipeline_question_with_llm(
#     question="What should the team merge into the monthly forecasting model?",
#     vector_store=vector_store,
#     model_name="gpt-4o-mini",
#     k=8
# )